# Rider Voice Decisions · live Gradio demo (free 2x T4)

Host the dispatch-console app on a **free Kaggle 2x T4** and get a public Gradio link you can open on your phone or send to someone watching the demo. You run this notebook yourself. Nothing is uploaded for you.

## Before you press Run All

1. In the notebook settings (right sidebar): set **Accelerator** to **GPU T4 x2** (two 15 GB cards; the ~16 GB model is split across both, it does not fit on one) and turn **Internet** **ON**.
2. **Add-ons → Secrets** → add a secret named exactly `HF_TOKEN`. Paste a **read** token from the [sahildl](https://huggingface.co/sahildl) Hugging Face account that can access the private model repo [`sahildl/rider-voice-decisions-m1v3`](https://huggingface.co/sahildl/rider-voice-decisions-m1v3). Do not paste the token into any cell, and do not use a write token.
3. **Run All**. The last code cell stays running and prints a public `*.gradio.live` link in large type.

The first run takes several minutes: it installs a couple of packages, then downloads the public ~16 GB `Darkrider/nirnaya-flash-1.0` backbone and the private m1v3 head. After warmup, a clip should land in about 0.6 s. The 16 GB weights sit tightly on a 16 GB T4; if you get a CUDA out-of-memory error, restart the session and run the notebook again.

This notebook is self-contained (the Space `sahildl/rider-voice-decisions` was not on the Hub when it was written), so it writes the app files into `/kaggle/working/rider-voice-space`. Weights are not baked in; they are downloaded with your token.

In [ ]:
# MUST run before anything imports transformers. Kaggle images ship TensorFlow;
# Gemma 4 / transformers 5.17 should not try to load it.
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient

_token = UserSecretsClient().get_secret("HF_TOKEN")
if not _token or not str(_token).strip():
    raise SystemExit(
        "Add a Kaggle secret named HF_TOKEN (a read token for the sahildl account). "
        "Add-ons → Secrets. Do not paste the token into a cell."
    )
os.environ["HF_TOKEN"] = _token
os.environ["HUGGING_FACE_HUB_TOKEN"] = _token
del _token
print("HF_TOKEN loaded from Kaggle secrets (value not printed).", flush=True)


In [ ]:
import importlib.metadata as md
import subprocess
import sys

def _run(cmd: str, check: bool = True) -> None:
    print("$", cmd, flush=True)
    proc = subprocess.run(cmd, shell=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f"failed ({proc.returncode}): {cmd}")

def _ver(name: str) -> str | None:
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

_run("nvidia-smi --query-gpu=name,memory.total --format=csv; free -g | head -2", check=False)

# Pin Gradio to the Space sdk_version (6.32.0). requirements.txt does not pin Gradio
# itself; the Space README does. Gemma 4 needs transformers==5.17.0. Do not reinstall torch.
need = []
if _ver("gradio") != "6.32.0":
    need.append("gradio==6.32.0")
if _ver("transformers") != "5.17.0":
    need.append("transformers==5.17.0")
for pkg in ("accelerate", "safetensors", "huggingface_hub", "librosa", "soundfile", "hf_transfer", "matplotlib", "edge-tts"):
    if _ver(pkg) is None:
        need.append(pkg)

if need:
    _run(f"{sys.executable} -m pip install -q " + " ".join(f"'{p}'" for p in need), check=True)
    print("installed:", ", ".join(need), flush=True)
else:
    print("all pinned packages already present", flush=True)

_run(
    f"{sys.executable} -c \"import gradio, transformers, torch; "
    f"print('gradio', gradio.__version__); print('transformers', transformers.__version__); "
    f"print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), "
    f"'gpus', torch.cuda.device_count())\"",
    check=True,
)


In [ ]:
from pathlib import Path

APP = Path("/kaggle/working/rider-voice-space")
for rel in ("src", "spec", "examples", "assets/sounds", "run"):
    (APP / rel).mkdir(parents=True, exist_ok=True)
print("app root:", APP, flush=True)


In [ ]:
%%writefile /kaggle/working/rider-voice-space/app.py
"""Rider Voice Decisions — dispatch-console demo (Gradio, ZeroGPU or local CPU).

    MOCK=1 python app.py          # no weights needed; for layout and screenshots
    python app.py                 # loads run/ (or $RUN_DIR / $RUN_ZIP)

Environment:
  MOCK=1                 skip the backbone and head; deterministic filename-based predictor
  RUN_DIR                trained head folder (default ./run): head.pt, config.json, t_entropy.json, thresholds.json
  RUN_ZIP                results zip containing out/runs/<RUN_NAME>/...; extracted into RUN_DIR on start
  RUN_NAME               run name inside the zip (default m1v3)
  HF_MODEL_REPO          Hub model repo to snapshot into RUN_DIR at startup (e.g. sahildl/rider-voice-decisions-m1v3)
  HF_TOKEN               read token for a private HF_MODEL_REPO (Space secret); never logged
  THRESHOLDS             optional thresholds json merged over the run's (default ./thresholds.json)
  MODEL_ID / MODEL_DIR   backbone (default Darkrider/nirnaya-flash-1.0)
  PRECISION              fp16 (default on CUDA) | bf16 | fp32 (CPU)
  NIRNAYA_DIR            nirnaya-inference checkout (cloned at a pinned commit when missing)
"""

from __future__ import annotations

import json
import os
import sys
import time
from io import BytesIO
from pathlib import Path

import gradio as gr
import matplotlib
import numpy as np
from PIL import Image

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

HERE = Path(__file__).resolve().parent
ROOT = HERE if (HERE / "src").exists() else HERE.parent
sys.path[:0] = [str(ROOT)]

from src.alert_rules import DEFAULTS, card, consistent  # noqa: E402
from src.rider_copy import rider_view  # noqa: E402

# --------------------------------------------------------------------------- config
def _truthy(v: str | None) -> bool:
    return (v or "").strip().lower() in {"1", "true", "yes", "on"}


def _load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


THRESHOLDS_PATH = Path(os.environ.get("THRESHOLDS", ROOT / "thresholds.json"))
BASE_THRESHOLDS = {**DEFAULTS, **_load_json(THRESHOLDS_PATH)}
CURVE_PATH = ROOT / "threshold_curve.json"
CURVE = _load_json(CURVE_PATH)

LABELS = {
    "situation": {
        "customer_unreachable": "Customer unreachable",
        "wrong_address": "Wrong address",
        "restaurant_delay": "Restaurant delay",
        "order_issue_at_pickup": "Order issue at pickup",
        "food_spilled_or_damaged": "Food spilled / damaged",
        "vehicle_breakdown": "Vehicle breakdown",
        "road_blocked_or_weather": "Road blocked / weather",
        "payment_issue": "Payment issue",
        "customer_dispute": "Customer dispute",
        "accident_or_injury": "Accident / injury",
        "harassment_or_threat": "Harassment / threat",
        "other_or_unclear": "Other / unclear",
    },
    "action": {
        "wait": "Wait here",
        "call_customer": "Call the customer",
        "contact_support": "Talk to support",
        "request_reassign": "Ask to reassign",
        "return_to_restaurant": "Go back to restaurant",
        "sos": "SOS",
    },
    "language": {"hinglish": "Hinglish", "hindi": "Hindi", "english": "English"},
}

SITUATIONS = list(LABELS["situation"])
ACTIONS = list(LABELS["action"])
LANGUAGES = list(LABELS["language"])

# Curated held-out synthetic test clips. Avoids the weak scripts
# wa_*_001, disp_hing_004, pay_hind_005, acc_*_010.
PRESETS = [
    ("SOS · Followed", "01_sos_being_followed_hinglish_10dB.wav"),
    ("SOS · Accident", "02_sos_accident_short_hinglish_10dB.wav"),
    ("Check-in · Safe now", "03_checkin_abused_now_safe_hindi_10dB.wav"),
    ("Unreachable · Gate locked", "04_call_customer_gate_locked_hinglish_10dB.wav"),
    ("Wait · On the way", "05_wait_customer_coming_hinglish_10dB.wav"),
    ("Support · Three calls", "06_support_three_calls_hinglish_10dB.wav"),
    ("Payment · Fetching cash", "07_wait_customer_fetching_cash_hindi_10dB.wav"),
    ("Dispute · Open packet", "08_support_dispute_open_packet_hinglish_10dB.wav"),
    ("Wrong pin", "09_call_customer_wrong_pin_hinglish_5dB.wav"),
]


def _preset_path(name: str) -> Path:
    return ROOT / "examples" / name


PRESET_PATHS = [(label, _preset_path(fn)) for label, fn in PRESETS if _preset_path(fn).exists()]

# --------------------------------------------------------------------------- thresholds / curve
def merge_thresholds(sos_tau: float | None = None) -> dict:
    th = dict(BASE_THRESHOLDS)
    if sos_tau is not None:
        th["sos_tau"] = float(sos_tau)
    return th


def lookup_curve(sos_tau: float) -> dict | None:
    pts = CURVE.get("curve") or []
    if not pts:
        return None
    return min(pts, key=lambda r: abs(float(r["sos_tau"]) - float(sos_tau)))


def _pct(x) -> str:
    if x is None:
        return "—"
    return f"{100 * float(x):.1f}%"


def curve_html(sos_tau: float) -> str:
    pt = lookup_curve(sos_tau)
    checkin = BASE_THRESHOLDS.get("checkin_tau", "—")
    if not pt:
        return (
            f'<div class="curve-pending">Curve pending — no held-out predictions were available when this '
            f"file was built. Run scripts/build_threshold_curve.py on a pred_test.jsonl to fill "
            f"threshold_curve.json. Check-in threshold is fixed at {checkin}.</div>"
        )
    src = CURVE.get("caption") or CURVE.get("source", "held-out eval")
    return (
        '<div class="curve-row">'
        f"<span>False SOS {_pct(pt['false_sos'])}</span>"
        f"<span>Safety recall {_pct(pt['safety_recall'])}</span>"
        f"<span>SOS recall (hard) {_pct(pt['sos_recall_hard'])}</span>"
        "</div>"
        f'<div class="curve-row"><span>Check-ins on normal {_pct(pt["checkin_on_normal"])}</span>'
        f"<span>Check-in τ {checkin}</span></div>"
        f'<div class="curve-source">{_esc(src)}</div>'
    )


# --------------------------------------------------------------------------- rendering
def _esc(s) -> str:
    return (
        str(s)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace('"', "&quot;")
    )


def banner_html(kind: str, latency_s: float | None, primary: str | None = None) -> str:
    if latency_s is None:
        meta = "⏱ —"
    elif latency_s < 0.05:
        meta = f"⏱ {max(latency_s, 0) * 1000:.1f} ms"
    else:
        meta = f"⏱ {latency_s:.2f} s"
    if kind == "sos":
        cls, label = "alert--sos", "SOS · RIDER UNSAFE · CALLING SUPPORT"
    elif kind == "checkin":
        cls, label = "alert--checkin", "ARE YOU SAFE? · CHECK-IN"
    elif kind == "result":
        act = LABELS["action"].get(primary or "", (primary or "no alert").replace("_", " "))
        cls, label = "alert--idle", f"NO ALERT · {act.upper()}"
        meta = meta + " · GPU" if os.environ.get("CUDA_VISIBLE_DEVICES") not in (None, "") else meta
    else:
        cls, label = "alert--idle", "LISTENING · NO ALERT"
    return f'<div class="alert {cls}"><div class="alert-label">{_esc(label)}</div><div class="alert-meta">{_esc(meta)}</div></div>'


def chips_html(answers: dict | None, kind: str | None = None) -> str:
    def one(q, raw):
        text = LABELS.get(q, {}).get(raw, (raw or "—").replace("_", " "))
        extra = " is-sos" if (q == "action" and raw == "sos") or kind == "sos" and q == "action" else ""
        extra = extra or (" is-empty" if not raw else "")
        return (
            f'<div class="chip-row"><div class="chip-key">{q}</div>'
            f'<div class="chip-val{extra}">{_esc(text)}</div></div>'
        )

    if not answers:
        return one("situation", None) + one("action", None) + one("language", None)
    return (
        one("situation", answers["situation"]["choice"])
        + one("action", answers["action"]["choice"])
        + one("language", answers["language"]["choice"])
    )


def confidence_html(answers: dict | None, kind: str | None = None) -> str:
    accent = " is-sos" if kind == "sos" else (" is-checkin" if kind == "checkin" else "")
    rows = []
    heads = [("situation", "situation"), ("action", "action"), ("language", "language")]
    for label, q in heads:
        if answers and q in answers and "probabilities" in answers[q]:
            p = answers[q]["probabilities"].get(answers[q]["choice"], 0.0)
        else:
            p = 0.0
        rows.append(
            f'<div class="conf-row"><div class="conf-name">{label}</div>'
            f'<div class="conf-track"><div class="conf-fill{accent}" style="width:{100 * p:.1f}%"></div></div>'
            f'<div class="conf-num">{p:.2f}</div></div>'
        )
    return '<div class="conf-block"><div class="conf-head">Confidence</div>' + "".join(rows) + "</div>"


LANG_TAG = {"hinglish": "HINGLISH", "hindi": "हिंदी", "english": "ENGLISH"}


def phone_html(view: dict | None) -> str:
    """The rider's phone after the model decides (illustrative; nothing is called or sent)."""
    if not view:
        body = (
            '<div class="ph-idle"><div class="ph-idle-dot"></div>'
            '<div class="ph-idle-t">Listening for a voice note…</div>'
            '<div class="ph-idle-s">Rider speaks once · the app answers here</div></div>'
        )
        cls, ops = "ph--idle", "Nothing sent yet"
    else:
        cls = {"sos": "ph--sos", "checkin": "ph--checkin"}.get(view["kind"], "ph--ok")
        btns = "".join(
            f'<div class="ph-btn{" ph-btn--primary" if i == 0 else ""}">{_esc(b)}</div>'
            for i, b in enumerate(view["buttons"])
        )
        then = f'<div class="ph-then">Then: {_esc(view["then"])}</div>' if view.get("then") else ""
        body = (
            f'<div class="ph-lang">{_esc(LANG_TAG.get(view["lang"], view["lang"]))}</div>'
            f'<div class="ph-line">{_esc(view["line"])}</div>'
            f'<div class="ph-gloss">{_esc(view["gloss"])}</div>'
            f'<div class="ph-say"><span class="ph-spk">🔊</span>“{_esc(view["spoken"])}”</div>'
            f'{then}<div class="ph-btns">{btns}</div>'
            f'<div class="ph-status">{_esc(view["status"])}</div>'
        )
        ops = view["ops"]
    return (
        '<div class="rider-wrap"><div class="rider-head"><span>RIDER SEES</span>'
        '<span class="rider-note">Illustrative rider experience · simulated, nothing is called or sent</span></div>'
        f'<div class="phone {cls}"><div class="ph-notch"></div><div class="ph-screen">{body}</div></div>'
        f'<div class="ops-line"><span class="ops-k">SENT TO OPS / CUSTOMER</span>{_esc(ops)}</div></div>'
    )


def _view_for(answers: dict, c: dict) -> dict:
    sit = LABELS["situation"].get(answers["situation"]["choice"], answers["situation"]["choice"])
    return rider_view(c, answers, sit.lower())


TTS_VOICE = {"english": "en-IN-NeerjaNeural", "hindi": "hi-IN-SwaraNeural", "hinglish": "hi-IN-SwaraNeural"}
TTS_DIR = Path(os.environ.get("TTS_CACHE", "/tmp/rider-tts"))


def speak(view: dict | None, muted: bool) -> str | None:
    """Spoken reply via edge-tts (needs internet). Returns None when muted, missing or offline."""
    if muted or not view or _truthy(os.environ.get("NO_TTS")):
        return None
    import hashlib

    voice = TTS_VOICE.get(view["lang"], "en-IN-NeerjaNeural")
    out = TTS_DIR / (hashlib.sha1(f"{voice}|{view['spoken']}".encode()).hexdigest()[:16] + ".mp3")
    if out.exists():
        return str(out)
    try:
        import asyncio

        import edge_tts

        TTS_DIR.mkdir(parents=True, exist_ok=True)

        async def _go():
            await asyncio.wait_for(edge_tts.Communicate(view["spoken"], voice).save(str(out)), 8)

        asyncio.run(_go())
        return str(out) if out.exists() and out.stat().st_size > 0 else None
    except Exception as exc:
        print(f"[rider-voice] tts skipped ({type(exc).__name__})", flush=True)
        return None


def counter_html(n: int) -> str:
    return (
        f'<div class="counter-block"><div class="counter-label">Check-ins this session</div>'
        f'<div class="counter-value">{int(n):02d}</div></div>'
    )


IDLE_SPEC = None


def _blank_spec() -> Image.Image:
    global IDLE_SPEC
    if IDLE_SPEC is None:
        img = Image.new("RGB", (1280, 240), (5, 5, 5))
        IDLE_SPEC = img
    return IDLE_SPEC


def spectrogram_image(path: str | None) -> Image.Image:
    if not path or not Path(path).exists():
        return _blank_spec()
    try:
        import soundfile as sf

        y, sr = sf.read(path, always_2d=False)
        if getattr(y, "ndim", 1) > 1:
            y = y.mean(axis=1)
        y = np.asarray(y, dtype=np.float32)
        if y.size < 16:
            return _blank_spec()
        nfft = 512
        hop = 128
        # Hann-windowed STFT
        window = np.hanning(nfft).astype(np.float32)
        if y.size < nfft:
            y = np.pad(y, (0, nfft - y.size))
        frames = 1 + (y.size - nfft) // hop
        # limit frames for a short console strip
        frames = max(1, min(frames, 400))
        spec = np.empty((nfft // 2 + 1, frames), dtype=np.float32)
        for i in range(frames):
            sl = y[i * hop : i * hop + nfft]
            if sl.size < nfft:
                sl = np.pad(sl, (0, nfft - sl.size))
            spec[:, i] = np.abs(np.fft.rfft(sl * window))
        db = 20.0 * np.log10(np.maximum(spec, 1e-8))
        db = db - db.max()
        fig, ax = plt.subplots(figsize=(10.5, 1.7), dpi=120, facecolor="#050505")
        ax.set_facecolor("#050505")
        ax.imshow(db, aspect="auto", origin="lower", cmap="magma", interpolation="nearest", vmin=-80, vmax=0)
        ax.axis("off")
        fig.subplots_adjust(0, 0, 1, 1)
        buf = BytesIO()
        fig.savefig(buf, format="png", facecolor="#050505", bbox_inches="tight", pad_inches=0)
        plt.close(fig)
        buf.seek(0)
        return Image.open(buf).convert("RGB")
    except Exception:
        return _blank_spec()


# --------------------------------------------------------------------------- mock + real engines
def _probs(keys: list[str], choice: str, p: float) -> dict[str, float]:
    rest = (1.0 - p) / max(len(keys) - 1, 1)
    return {k: (p if k == choice else rest) for k in keys}


# Filename keywords -> a deterministic, schema-identical decision (used only when MOCK=1
# or when no head.pt is present). These are staged examples, not model outputs.
_MOCK_BY_PREFIX = {
    "01_sos_being_followed": ("harassment_or_threat", "sos", "hinglish", 0.93, 0.81, 0.88),
    "02_sos_accident_short": ("accident_or_injury", "sos", "hinglish", 0.91, 0.86, 0.79),
    "03_checkin_abused_now_safe": ("harassment_or_threat", "contact_support", "hindi", 0.84, 0.07, 0.72),
    "04_call_customer_gate_locked": ("customer_unreachable", "call_customer", "hinglish", 0.89, 0.01, 0.02),
    "05_wait_customer_coming": ("customer_unreachable", "wait", "hinglish", 0.82, 0.01, 0.02),
    "06_support_three_calls": ("customer_unreachable", "contact_support", "hinglish", 0.80, 0.02, 0.03),
    "07_wait_customer_fetching_cash": ("payment_issue", "wait", "hindi", 0.86, 0.01, 0.01),
    "08_support_dispute_open_packet": ("customer_dispute", "contact_support", "hinglish", 0.78, 0.02, 0.03),
    "09_call_customer_wrong_pin": ("wrong_address", "call_customer", "hinglish", 0.77, 0.01, 0.02),
}


def _mock_tuple(path: str | None) -> tuple:
    stem = Path(path or "").name
    for key, val in _MOCK_BY_PREFIX.items():
        if stem.startswith(key):
            return val
    return ("other_or_unclear", "contact_support", "english", 0.55, 0.02, 0.03)


def mock_decide(path: str, th: dict) -> dict:
    sit, act, lang, p_top, p_sos, p_safe = _mock_tuple(path)
    sit_p = _probs(SITUATIONS, sit, p_top)
    # keep a visible safety-situation mass on the check-in / SOS presets
    if sit in {"accident_or_injury", "harassment_or_threat"}:
        sit_p[sit] = max(sit_p[sit], p_safe)
        # renormalize roughly
        z = sum(sit_p.values()) or 1.0
        sit_p = {k: v / z for k, v in sit_p.items()}
    act_p = _probs(ACTIONS, act, p_top)
    act_p["sos"] = p_sos
    # renormalize actions after forcing p_sos
    others = [a for a in ACTIONS if a != "sos"]
    remain = max(0.0, 1.0 - p_sos)
    top_share = p_top if act != "sos" else 0.0
    rest = remain - top_share
    if act != "sos":
        act_p[act] = top_share
        spread = [a for a in others if a != act]
        for a in spread:
            act_p[a] = (rest / len(spread)) if spread else 0.0
    else:
        for a in others:
            act_p[a] = remain / len(others)
    s = sum(act_p.values()) or 1.0
    act_p = {k: v / s for k, v in act_p.items()}
    lang_p = _probs(LANGUAGES, lang, max(p_top, 0.7))
    answers = {
        "situation": {"type": "choice", "choice": sit, "probabilities": sit_p},
        "action": {"type": "choice", "choice": act, "probabilities": act_p},
        "safety": {"type": "noul", "noul": p_safe},
        "language": {"type": "choice", "choice": lang, "probabilities": lang_p},
    }
    answers, rep = consistent(answers, th)
    return {
        "answers": answers,
        "consistency": rep,
        "card": card(answers, th),
        "precision": "mock",
        "audio_s": None,
        "n_tokens": 0,
    }


def _want_mock() -> bool:
    if _truthy(os.environ.get("MOCK")):
        return True
    run = Path(os.environ.get("RUN_DIR", ROOT / "run"))
    zip_path = os.environ.get("RUN_ZIP")
    repo = (os.environ.get("HF_MODEL_REPO") or "").strip()
    return not (run / "head.pt").exists() and not zip_path and not repo


MOCK = _want_mock()
ENGINE = None
ENGINE_ERROR = None


def _extract_run_zip(zip_path: Path, run_name: str, dst: Path) -> None:
    import shutil
    import zipfile

    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        for f in ("head.pt", "config.json", "t_entropy.json", "thresholds.json"):
            hits = [n for n in names if n.endswith(f"/runs/{run_name}/{f}")]
            if not hits:
                hits = [n for n in names if f"/{run_name}/{f}" in n or n.endswith(f"/{run_name}/{f}")]
            if hits:
                with z.open(hits[0]) as r, open(dst / f, "wb") as w:
                    shutil.copyfileobj(r, w)


def _download_hf_model_repo(dst: Path) -> None:
    """Snapshot the trained head from HF_MODEL_REPO into RUN_DIR. Uses HF_TOKEN if set; never logs it."""
    repo = (os.environ.get("HF_MODEL_REPO") or "").strip()
    if not repo:
        return
    if (dst / "head.pt").exists() and (dst / "config.json").exists():
        return
    from huggingface_hub import snapshot_download

    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    dst.mkdir(parents=True, exist_ok=True)
    print(f"[rider-voice] downloading run files from {repo} -> {dst}", flush=True)
    snapshot_download(
        repo_id=repo,
        repo_type="model",
        local_dir=str(dst),
        token=token,
        allow_patterns=[
            "head.pt",
            "config.json",
            "t_entropy.json",
            "thresholds.json",
            "calibration.json",
        ],
    )


def _resolve_run_dir() -> Path:
    dst = Path(os.environ.get("RUN_DIR", ROOT / "run"))
    z = os.environ.get("RUN_ZIP")
    name = os.environ.get("RUN_NAME", "m1v3")
    if z and not (dst / "head.pt").exists():
        _extract_run_zip(Path(z), name, dst)
    if not (dst / "head.pt").exists():
        _download_hf_model_repo(dst)
    return dst


def _gpu_decorator():
    try:  # ZeroGPU: import `spaces` before torch; GPU is attached only inside @spaces.GPU
        import spaces  # type: ignore

        return spaces.GPU(duration=40)
    except ImportError:
        def passthrough(f):
            return f

        return passthrough


gpu = _gpu_decorator()


def _load_engine():
    global ENGINE, ENGINE_ERROR, BASE_THRESHOLDS
    if MOCK:
        return None
    try:
        import subprocess

        nirnaya = Path(os.environ.get("NIRNAYA_DIR", ROOT / "nirnaya-inference"))
        commit = "0fcd64f22b3013dd2bb1e268533bcc91accf87c6"
        if not (nirnaya / "mmclm").exists():
            subprocess.run(["git", "clone", "https://github.com/NeverInAsh/nirnaya-inference", str(nirnaya)], check=True)
            subprocess.run(["git", "-C", str(nirnaya), "checkout", commit], check=True)
        sys.path[:0] = [str(nirnaya)]
        import torch  # noqa: E402
        from src.engine import RiderVoiceEngine  # noqa: E402

        if os.environ.get("MODEL_DIR"):
            model = os.environ["MODEL_DIR"]
        else:
            from huggingface_hub import snapshot_download

            model = snapshot_download(os.environ.get("MODEL_ID", "Darkrider/nirnaya-flash-1.0"))
        cuda = torch.cuda.is_available()
        precision = os.environ.get("PRECISION", "fp16" if cuda else "fp32")
        run = _resolve_run_dir()
        th = dict(BASE_THRESHOLDS)
        run_th = _load_json(run / "thresholds.json")
        # run file first, then the Space defaults / THRESHOLDS override (round-3 two-level)
        merged = {**run_th, **th}
        BASE_THRESHOLDS = merged
        ENGINE = RiderVoiceEngine(
            model,
            str(run),
            device="cuda" if cuda else "cpu",
            precision=precision,
            # one big GPU (>=20 GB) holds the ~16 GB fp16 model; otherwise split over every visible GPU (Kaggle 2x T4)
            placement="single" if cuda and torch.cuda.get_device_properties(0).total_memory >= 20 * 2**30 else "auto",
            thresholds=merged,
        )
        return ENGINE
    except Exception as exc:  # weights or backbone missing: stay in mock rather than crash the Space
        ENGINE_ERROR = f"{type(exc).__name__}: {exc}"
        print(f"[rider-voice] real engine failed to load ({ENGINE_ERROR}); using MOCK predictor", flush=True)
        return None


@gpu
def _infer(path: str, th: dict) -> dict:
    if ENGINE is not None:
        # The engine already applies the run's thresholds; re-apply `th` so the slider can retune SOS.
        out = ENGINE.decide(path)
        ans, rep = consistent(out["answers"], th)
        out = {**out, "answers": ans, "consistency": rep, "card": card(ans, th)}
        return out
    return mock_decide(path, th)


def _warmup():
    if ENGINE is None:
        return
    try:
        import soundfile as sf

        silence = ROOT / "run" / "_warmup_silence.wav"
        silence.parent.mkdir(parents=True, exist_ok=True)
        if not silence.exists():
            sr = 16000
            sf.write(str(silence), np.zeros(int(0.4 * sr), dtype=np.float32), sr)
        _infer(str(silence), merge_thresholds())
        print("[rider-voice] warmup done", flush=True)
    except Exception as exc:
        print(f"[rider-voice] warmup skipped ({type(exc).__name__})", flush=True)


# --------------------------------------------------------------------------- sound hook (licensed files only; silent if absent)
def _sound_for(kind: str, muted: bool) -> str | None:
    if muted or kind not in {"sos", "checkin"}:
        return None
    p = ROOT / "assets" / "sounds" / f"{kind}.wav"
    return str(p) if p.exists() else None


# --------------------------------------------------------------------------- predict
def _mode_note() -> str:
    if ENGINE is not None:
        prec = getattr(ENGINE, "bb", None)
        prec = getattr(prec, "precision", "fp32") if prec is not None else "fp32"
        return f'<div class="mode-chip">run · {prec}</div>'
    return '<div class="mode-chip">demo mode · mock predictor (weights not loaded)</div>'


def predict(audio_path, checkins, muted, sos_tau, last_out):
    th = merge_thresholds(sos_tau)
    metrics = curve_html(float(sos_tau)) + _mode_note()
    if not audio_path:
        return (
            banner_html("idle", None),
            _blank_spec(),
            chips_html(None),
            confidence_html(None),
            metrics,
            counter_html(int(checkins or 0)),
            int(checkins or 0),
            None,
            last_out,
            phone_html(None),
            None,
        )
    t0 = time.perf_counter()
    out = _infer(audio_path, th)
    # Time only the infer call. Prefer the engine's own counter when it ran the model.
    if ENGINE is not None and "latency_ms" in out:
        latency_s = float(out["latency_ms"]) / 1000.0
    else:
        latency_s = time.perf_counter() - t0
    kind = out["card"]["kind"]
    banner_kind = kind if kind in {"sos", "checkin"} else "result"
    new_checkins = int(checkins or 0)
    if kind == "checkin":
        new_checkins += 1
    view = _view_for(out["answers"], out["card"])
    packed = json.dumps(
        {"answers": out["answers"], "card": out["card"], "latency_s": latency_s, "path": audio_path},
        ensure_ascii=False,
    )
    return (
        banner_html(banner_kind, latency_s, out["answers"]["action"]["choice"]),
        spectrogram_image(audio_path),
        chips_html(out["answers"], kind),
        confidence_html(out["answers"], kind),
        metrics,
        counter_html(new_checkins),
        new_checkins,
        _sound_for(kind, bool(muted)),
        packed,
        phone_html(view),
        speak(view, bool(muted)),
    )


def retune(sos_tau, last_out, checkins):
    """Slider: look up real curve numbers; re-apply the alert rule to the last answers without re-running the model."""
    th = merge_thresholds(sos_tau)
    metrics = curve_html(float(sos_tau)) + _mode_note()
    if not last_out:
        return banner_html("idle", None), chips_html(None), confidence_html(None), metrics, counter_html(int(checkins or 0)), phone_html(None)
    packed = json.loads(last_out)
    answers, rep = consistent(packed["answers"], th)
    c = card(answers, th)
    kind = c["kind"]
    banner_kind = kind if kind in {"sos", "checkin"} else "result"
    return (
        banner_html(banner_kind, packed.get("latency_s"), answers["action"]["choice"]),
        chips_html(answers, kind),
        confidence_html(answers, kind),
        metrics,
        counter_html(int(checkins or 0)),
        phone_html(_view_for(answers, c)),
    )


def toggle_demo(playing: bool):
    playing = not bool(playing)
    label = "Stop demo" if playing else "Autoplay demo"
    return playing, gr.Timer(value=5, active=playing), label


def on_timer(idx, checkins, muted, sos_tau, last_out, playing):
    if not playing or not PRESET_PATHS:
        return (int(idx or 0),) + (gr.update(),) * 12
    n = len(PRESET_PATHS)
    idx = int(idx or 0) % n
    path = str(PRESET_PATHS[idx][1])
    idx = (idx + 1) % n
    outs = predict(path, checkins, muted, sos_tau, last_out)
    return (idx, path, *outs)


def on_preset(path, checkins, muted, sos_tau, last_out):
    return (path, *predict(path, checkins, muted, sos_tau, last_out))


# --------------------------------------------------------------------------- UI
CSS = (ROOT / "style.css").read_text(encoding="utf-8") if (ROOT / "style.css").exists() else ""
DARK_JS = """
() => {
  document.documentElement.classList.add('dark');
  document.body.classList.add('dark');
}
"""
HEAD = """
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=JetBrains+Mono:wght@400;500;600&display=swap" rel="stylesheet">
"""

DEFAULT_SOS = float(BASE_THRESHOLDS.get("sos_tau", 0.42))

theme = gr.themes.Base(
    primary_hue="red",
    secondary_hue="zinc",
    neutral_hue="zinc",
    font=[gr.themes.GoogleFont("Space Grotesk"), "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "monospace"],
    radius_size="sm",
).set(
    body_background_fill="#050505",
    body_text_color="#EDEDED",
    block_background_fill="#0D0D0F",
    block_border_color="#232326",
    block_radius="6px",
    border_color_primary="#232326",
    button_primary_background_fill="#FF2D2D",
    button_primary_text_color="#050505",
    slider_color="#FF2D2D",
)

with gr.Blocks(title="Rider Voice Decisions", fill_height=True, fill_width=True, elem_id="dispatch") as demo:
    checkins = gr.State(0)
    last_out = gr.State("")
    playing = gr.State(False)
    demo_idx = gr.State(0)
    timer = gr.Timer(value=5, active=False)

    banner = gr.HTML(banner_html("idle", None), elem_id="alert-banner")

    with gr.Row(elem_id="main-row", equal_height=True):
        with gr.Column(scale=4, elem_id="left-col"):
            audio = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="VOICE NOTE",
                elem_id="audio-in",
                waveform_options=gr.WaveformOptions(
                    waveform_color="#4A4A4F",
                    waveform_progress_color="#FF2D2D",
                    show_recording_waveform=True,
                ),
            )
            with gr.Column(elem_id="presets-wrap"):
                gr.Markdown("PRESETS", elem_id="presets-label")
                with gr.Row(elem_id="preset-grid"):
                    preset_buttons = []
                    for label, path in PRESET_PATHS:
                        preset_buttons.append(
                            (gr.Button(label, elem_classes=["preset-btn"], size="sm"), str(path))
                        )
            with gr.Row(elem_id="controls-row"):
                demo_btn = gr.Button("Autoplay demo", elem_id="demo-toggle", size="sm")
                mute = gr.Checkbox(label="Mute sound", value=True, elem_id="mute-toggle")
                count_html = gr.HTML(counter_html(0), elem_id="checkin-counter")
            sound = gr.Audio(label="alert", autoplay=True, visible=False, elem_id="hidden-sound")

        with gr.Column(scale=5, elem_id="phone-col"):
            phone = gr.HTML(phone_html(None), elem_id="rider-phone")
            voice = gr.Audio(label="spoken reply", autoplay=True, visible=False, elem_id="hidden-voice")

        with gr.Column(scale=4, elem_id="right-col"):
            chips = gr.HTML(chips_html(None), elem_id="chips")
            slider = gr.Slider(
                minimum=0.01,
                maximum=0.99,
                step=0.01,
                value=DEFAULT_SOS,
                label="SOS THRESHOLD",
                info="Held-out eval curve. Check-in threshold stays fixed.",
                elem_id="sos-slider",
            )
            metrics = gr.HTML(curve_html(DEFAULT_SOS) + _mode_note(), elem_id="curve-panel")
            with gr.Column(elem_id="internals"):
                gr.Markdown("MODEL INTERNALS", elem_id="internals-label")
                conf = gr.HTML(confidence_html(None), elem_id="confidence")
                spec = gr.Image(
                    value=_blank_spec(),
                    label="SPECTROGRAM",
                    elem_id="spectrogram",
                    show_label=True,
                    interactive=False,
                    type="pil",
                    format="png",
                    buttons=[],
                )

    gr.HTML(
        '<div class="site-footer">Inspired by delivery apps · Not affiliated with Zomato or Swiggy · '
        "Synthetic voices, no real rider data · Built by Sahil Builds AI</div>",
        elem_id="site-footer",
    )

    predict_inputs = [audio, checkins, mute, slider, last_out]
    predict_outputs = [banner, spec, chips, conf, metrics, count_html, checkins, sound, last_out, phone, voice]

    audio.stop_recording(predict, predict_inputs, predict_outputs)
    audio.upload(predict, predict_inputs, predict_outputs)

    for btn, path in preset_buttons:
        btn.click(
            lambda checkins, muted, sos_tau, last, p=path: (p, *predict(p, checkins, muted, sos_tau, last)),
            inputs=[checkins, mute, slider, last_out],
            outputs=[audio, *predict_outputs],
        )

    slider.release(retune, [slider, last_out, checkins], [banner, chips, conf, metrics, count_html, phone])
    slider.change(lambda t: curve_html(float(t)) + _mode_note(), slider, metrics)

    demo_btn.click(toggle_demo, playing, [playing, timer, demo_btn])
    timer.tick(on_timer, [demo_idx, checkins, mute, slider, last_out, playing], [demo_idx, audio, *predict_outputs])


def _boot():
    _load_engine()
    # Warm the real model when a GPU is already visible (local or a persistent Space).
    # On ZeroGPU the device is only attached inside @spaces.GPU, so the first request warms it there.
    try:
        import torch

        if ENGINE is not None and torch.cuda.is_available():
            _warmup()
    except Exception:
        pass


_boot()


if __name__ == "__main__":
    css_path = ROOT / "style.css"
    demo.launch(
        theme=theme,
        css=CSS,
        js=DARK_JS,
        head=HEAD,
        footer_links=[],
        server_name="0.0.0.0",
        server_port=int(os.environ.get("PORT", "7860")),
        allowed_paths=[str(ROOT / "examples"), str(ROOT / "assets"), str(ROOT)],
        ssr_mode=False,
        show_error=True,
    )


In [ ]:
%%writefile /kaggle/working/rider-voice-space/style.css
@import url("https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=JetBrains+Mono:wght@400;500;600&display=swap");

:root {
  --bg: #050505;
  --surface: #0d0d0f;
  --border: #232326;
  --text: #ededed;
  --muted: #8a8a8f;
  --dim: #4a4a4f;
  --red: #ff2d2d;
  --amber: #ffb020;
  --idle: #1a1a1c;
  --radius: 6px;
}

html, body {
  background: var(--bg) !important;
  color: var(--text);
  height: 100%;
  overflow: hidden !important;
}

.gradio-container {
  max-width: 100% !important;
  width: 100% !important;
  height: 100% !important;
  min-height: 100vh;
  max-height: 100vh;
  padding: 12px 16px 8px !important;
  overflow: hidden !important;
  background-color: var(--bg) !important;
  background-image:
    linear-gradient(rgba(255, 255, 255, 0.05) 1px, transparent 1px),
    linear-gradient(90deg, rgba(255, 255, 255, 0.05) 1px, transparent 1px);
  background-size: 24px 24px;
  font-family: "Space Grotesk", sans-serif;
}

.gradio-container .contain,
#dispatch,
.main,
.wrap {
  background: transparent !important;
}

/* Hide Gradio's default footer and built-in chrome we replace */
footer,
footer.svelte-1axqjz2,
.footer,
.gradio-container footer {
  display: none !important;
}

#dispatch {
  height: 100%;
  max-height: calc(100vh - 4px);
  overflow: hidden;
}

/* ----- alert banner ----- */
#alert-banner {
  margin: 0 0 10px 0 !important;
}
#alert-banner .html-container,
#alert-banner .prose {
  padding: 0 !important;
  margin: 0 !important;
}
.alert {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 16px;
  min-height: 56px;
  padding: 0 20px;
  border-radius: var(--radius);
  border: 1px solid var(--border);
  box-sizing: border-box;
  overflow: hidden;
}
.alert-label {
  font-family: "Space Grotesk", sans-serif;
  font-size: 26px;
  font-weight: 700;
  letter-spacing: 0.08em;
  text-transform: uppercase;
  line-height: 1.1;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}
.alert-meta {
  font-family: "JetBrains Mono", monospace;
  font-size: 16px;
  font-weight: 500;
  white-space: nowrap;
  flex-shrink: 0;
}
.alert--idle {
  background: var(--idle);
  color: var(--muted);
  border-color: var(--border);
}
.alert--checkin {
  background: #1a1408;
  color: var(--amber);
  border-color: var(--amber);
}
.alert--sos {
  background: #1a0505;
  color: var(--red);
  border-color: var(--red);
  box-shadow: 0 0 24px rgba(255, 45, 45, 0.35);
  animation: sos-pulse 1.4s ease-in-out infinite;
}
@keyframes sos-pulse {
  0%, 100% { box-shadow: 0 0 24px rgba(255, 45, 45, 0.35); }
  50% { box-shadow: 0 0 40px rgba(255, 45, 45, 0.65); }
}

/* ----- columns ----- */
#main-row {
  gap: 16px !important;
  height: calc(100vh - 108px) !important;
  max-height: calc(100vh - 108px) !important;
  overflow: hidden !important;
  align-items: stretch !important;
}
#left-col, #right-col, #phone-col {
  background: var(--surface);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  padding: 10px 12px 8px !important;
  overflow: hidden !important;
  min-width: 0;
}

/* ----- audio / spectrogram ----- */
#audio-in, #audio-in .block, #audio-in .styler {
  background: transparent !important;
  border: none !important;
  box-shadow: none !important;
  padding: 0 !important;
}
#audio-in label,
#spectrogram label,
#sos-slider label,
#presets-wrap label {
  font-family: "JetBrains Mono", monospace !important;
  font-size: 11px !important;
  letter-spacing: 0.12em;
  text-transform: uppercase;
  color: var(--muted) !important;
}
#audio-in .waveform,
#audio-in waveform {
  background: #050505 !important;
}
#audio-in .source-selection { display: none !important; }
#audio-in .audio-container { max-height: 170px; overflow: hidden; }
#audio-in .component-wrapper { max-height: 140px; }

#spectrogram {
  margin-top: 4px !important;
}
#spectrogram .image-container,
#spectrogram .image-frame,
#spectrogram button.svelte-12vrxzd,
#spectrogram button {
  width: 100% !important;
  height: 110px !important;
  max-height: 110px !important;
  display: block !important;
  padding: 0 !important;
  background: #050505 !important;
  border: none !important;
}
#spectrogram img {
  width: 100% !important;
  height: 110px !important;
  max-height: 110px !important;
  object-fit: cover !important;
  display: block !important;
  opacity: 1 !important;
  border-radius: var(--radius);
  border: 1px solid var(--border);
  background: #050505 !important;
}
#spectrogram .icon-button-wrapper,
#spectrogram .icon-button,
#spectrogram .download,
#spectrogram .fullscreen {
  display: none !important;
}

/* ----- presets ----- */
#presets-wrap {
  margin-top: 6px !important;
}
#preset-grid {
  display: flex;
  flex-wrap: wrap;
  gap: 6px;
}
#preset-grid button,
.preset-btn {
  font-family: "JetBrains Mono", monospace !important;
  font-size: 11px !important;
  letter-spacing: 0.04em;
  text-transform: uppercase;
  background: #050505 !important;
  color: var(--text) !important;
  border: 1px solid var(--border) !important;
  border-radius: var(--radius) !important;
  padding: 6px 10px !important;
  min-height: 32px !important;
  box-shadow: none !important;
}
#preset-grid button:hover,
.preset-btn:hover {
  border-color: var(--red) !important;
  color: var(--red) !important;
}

/* ----- controls row ----- */
#controls-row {
  align-items: center !important;
  margin-top: 8px !important;
  gap: 8px !important;
}
#controls-row button,
#demo-toggle,
#mute-toggle {
  font-family: "JetBrains Mono", monospace !important;
  font-size: 12px !important;
  letter-spacing: 0.08em;
  text-transform: uppercase;
  background: #050505 !important;
  color: var(--text) !important;
  border: 1px solid var(--border) !important;
  border-radius: var(--radius) !important;
  box-shadow: none !important;
  min-height: 36px !important;
}
#demo-toggle[aria-pressed="true"],
#demo-toggle.active {
  border-color: var(--red) !important;
  color: var(--red) !important;
}

#checkin-counter .html-container,
#checkin-counter .prose {
  padding: 0 !important;
  margin: 0 !important;
}
.counter-block {
  text-align: right;
  padding: 0 4px;
}
.counter-label {
  font-family: "JetBrains Mono", monospace;
  font-size: 10px;
  letter-spacing: 0.14em;
  text-transform: uppercase;
  color: var(--muted);
}
.counter-value {
  font-family: "JetBrains Mono", monospace;
  font-size: 56px;
  font-weight: 600;
  line-height: 1;
  color: var(--text);
  letter-spacing: -0.04em;
}

/* ----- chips ----- */
#chips .html-container,
#confidence .html-container,
#curve-panel .html-container,
#chips .prose,
#confidence .prose,
#curve-panel .prose {
  padding: 0 !important;
  margin: 0 !important;
}
.chip-row {
  display: grid;
  grid-template-columns: 110px 1fr;
  gap: 10px;
  align-items: center;
  margin-bottom: 10px;
}
.chip-key {
  font-family: "JetBrains Mono", monospace;
  font-size: 11px;
  letter-spacing: 0.14em;
  text-transform: uppercase;
  color: var(--muted);
}
.chip-val {
  font-family: "Space Grotesk", sans-serif;
  font-size: 22px;
  font-weight: 600;
  letter-spacing: 0.04em;
  text-transform: uppercase;
  color: var(--text);
  background: #050505;
  border: 1px solid var(--border);
  border-radius: var(--radius);
  padding: 8px 14px;
  line-height: 1.2;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}
.chip-val.is-empty {
  color: var(--dim);
}
.chip-val.is-sos {
  color: var(--red);
  border-color: var(--red);
}

/* ----- confidence bars ----- */
.conf-block { margin-top: 4px; }
.conf-head {
  font-family: "JetBrains Mono", monospace;
  font-size: 11px;
  letter-spacing: 0.14em;
  text-transform: uppercase;
  color: var(--muted);
  margin-bottom: 8px;
}
.conf-row {
  display: grid;
  grid-template-columns: 110px 1fr 52px;
  gap: 10px;
  align-items: center;
  margin-bottom: 8px;
}
.conf-name {
  font-family: "JetBrains Mono", monospace;
  font-size: 12px;
  color: var(--muted);
  text-transform: uppercase;
  letter-spacing: 0.06em;
}
.conf-track {
  height: 8px;
  background: #050505;
  border: 1px solid var(--border);
  border-radius: 2px;
  overflow: hidden;
}
.conf-fill {
  height: 100%;
  background: var(--text);
  width: 0%;
}
.conf-fill.is-sos { background: var(--red); }
.conf-fill.is-checkin { background: var(--amber); }
.conf-num {
  font-family: "JetBrains Mono", monospace;
  font-size: 14px;
  text-align: right;
  color: var(--text);
}

/* ----- threshold slider ----- */
#sos-slider input[type="range"],
#sos-slider .svelte-range {
  accent-color: var(--red);
}
#curve-panel { margin-top: 4px; }
.curve-row {
  display: flex;
  justify-content: space-between;
  gap: 16px;
  font-family: "JetBrains Mono", monospace;
  font-size: 15px;
  color: var(--text);
  margin-top: 6px;
}
.curve-pending {
  color: var(--amber);
  font-family: "JetBrains Mono", monospace;
  font-size: 14px;
  margin-top: 8px;
}
.curve-source {
  font-family: "JetBrains Mono", monospace;
  font-size: 10px;
  color: var(--dim);
  margin-top: 8px;
  line-height: 1.4;
}
.mode-chip {
  font-family: "JetBrains Mono", monospace;
  font-size: 10px;
  letter-spacing: 0.12em;
  text-transform: uppercase;
  color: var(--dim);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  padding: 2px 8px;
  display: inline-block;
  margin-top: 6px;
}

/* ----- footer ----- */
#site-footer .html-container,
#site-footer .prose {
  padding: 0 !important;
  margin: 0 !important;
}
#site-footer {
  position: fixed !important;
  left: 0;
  right: 0;
  bottom: 0;
  height: 28px;
  max-height: 28px;
  overflow: hidden;
  z-index: 40;
  background: #050505;
  border-top: 1px solid #232326;
}
#left-col, #right-col, #phone-col { padding-bottom: 36px !important; }
.site-footer {
  font-family: "JetBrains Mono", monospace;
  font-size: 11px;
  color: var(--muted);
  letter-spacing: 0.02em;
  text-align: center;
  padding: 6px 8px 4px;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}

/* Tighten Gradio blocks */
.block, .form, .padded, .styler {
  background: transparent !important;
  border: none !important;
  box-shadow: none !important;
}
label.svelte-1b6s6s, .label-wrap {
  color: var(--muted) !important;
}
#hidden-sound, #hidden-state, #last-out {
  display: none !important;
  height: 0 !important;
  overflow: hidden !important;
}

/* Slider label typography */
#sos-slider span, #sos-slider .suffix {
  font-family: "JetBrains Mono", monospace !important;
  color: var(--text) !important;
}

/* ----- rider phone (hero) ----- */
#phone-col { display: flex; align-items: center; }
#rider-phone, #rider-phone .html-container, #rider-phone .prose { padding: 0 !important; margin: 0 !important; width: 100%; height: 100%; }
.rider-wrap { display: flex; flex-direction: column; align-items: center; height: 100%; gap: 8px; }
.rider-head { width: 100%; display: flex; justify-content: space-between; align-items: baseline; gap: 8px;
  font-family: "JetBrains Mono", monospace; font-size: 13px; letter-spacing: .12em; color: var(--text); font-weight: 600; }
.rider-note { font-size: 10px; letter-spacing: .02em; color: var(--muted); font-weight: 400; text-align: right; }
.phone { position: relative; flex: 1 1 auto; min-height: 0; aspect-ratio: 9 / 16; max-height: 600px; width: auto;
  border: 2px solid #2E2E33; border-radius: 34px; background: #000; padding: 10px; box-sizing: border-box; }
.ph-notch { position: absolute; top: 12px; left: 50%; transform: translateX(-50%); width: 34%; height: 18px; background: #0D0D0F; border-radius: 0 0 12px 12px; z-index: 2; }
.ph-screen { height: 100%; border-radius: 26px; background: #0D0D0F; border: 1px solid var(--border);
  display: flex; flex-direction: column; padding: 38px 16px 16px; box-sizing: border-box; overflow: hidden; }
.ph-lang { align-self: flex-start; font-family: "JetBrains Mono", monospace; font-size: 11px; letter-spacing: .1em;
  color: var(--muted); border: 1px solid var(--border); border-radius: 4px; padding: 2px 8px; }
.ph-line { font-family: "Space Grotesk", sans-serif; font-weight: 700; font-size: 30px; line-height: 1.12; color: var(--text); margin-top: 14px; }
.ph-gloss { font-size: 14px; color: var(--muted); margin-top: 6px; }
.ph-say { margin-top: 14px; font-size: 13px; line-height: 1.35; color: #C9C9CE; background: #050505; border: 1px solid var(--border);
  border-radius: 6px; padding: 8px 10px; }
.ph-spk { margin-right: 6px; }
.ph-then { margin-top: 8px; font-family: "JetBrains Mono", monospace; font-size: 11px; color: var(--muted); }
.ph-btns { margin-top: auto; display: flex; flex-direction: column; gap: 10px; }
.ph-btn { font-family: "Space Grotesk", sans-serif; font-weight: 700; font-size: 20px; text-align: center; padding: 16px 10px;
  border-radius: 12px; border: 1px solid #3A3A3F; color: var(--text); background: #18181B; }
.ph-btn--primary { background: var(--text); color: #050505; border-color: var(--text); }
.ph-status { margin-top: 10px; font-family: "JetBrains Mono", monospace; font-size: 11px; color: var(--muted); text-align: center; }
.ph--checkin .ph-screen { border-color: var(--amber); background: linear-gradient(180deg, rgba(255,176,32,.16), #0D0D0F 55%); }
.ph--checkin .ph-line { color: var(--amber); font-size: 40px; }
.ph--checkin .ph-btn--primary { background: var(--amber); border-color: var(--amber); }
.ph--sos .ph-screen { background: var(--red); border-color: var(--red); animation: phpulse 1.2s ease-in-out infinite; }
.ph--sos .ph-line { color: #fff; font-size: 44px; }
.ph--sos .ph-gloss, .ph--sos .ph-lang, .ph--sos .ph-status { color: rgba(255,255,255,.85); border-color: rgba(255,255,255,.4); }
.ph--sos .ph-say { background: rgba(0,0,0,.25); border-color: rgba(255,255,255,.3); color: #fff; }
.ph--sos .ph-btn--primary { background: #050505; color: #fff; border-color: #050505; }
.ph--sos .phone, .phone.ph--sos { box-shadow: 0 0 24px rgba(255,45,45,.35); }
@keyframes phpulse { 0%,100% { filter: brightness(1); } 50% { filter: brightness(1.18); } }
.ph-idle { margin: auto 0; text-align: center; color: var(--muted); }
.ph-idle-dot { width: 14px; height: 14px; border-radius: 50%; background: var(--dim); margin: 0 auto 14px; }
.ph-idle-t { font-family: "Space Grotesk", sans-serif; font-size: 20px; color: var(--text); }
.ph-idle-s { font-size: 12px; margin-top: 6px; }
.ops-line { width: 100%; font-family: "JetBrains Mono", monospace; font-size: 11px; color: var(--text); line-height: 1.4;
  border: 1px solid var(--border); border-radius: 6px; padding: 6px 10px; background: #050505; }
.ops-k { display: block; color: var(--red); letter-spacing: .1em; font-size: 10px; margin-bottom: 2px; }

/* ----- compact model internals ----- */
#internals { margin-top: auto; border-top: 1px solid var(--border); padding-top: 6px !important; gap: 4px !important; }
#internals-label p { font-family: "JetBrains Mono", monospace; font-size: 10px; letter-spacing: .12em; color: var(--muted); margin: 0; }
#internals .conf-row { margin-bottom: 2px; }
#internals .conf-head { display: none; }
#spectrogram .image-container, #spectrogram .image-frame, #spectrogram button, #spectrogram img { height: 56px !important; max-height: 56px !important; }
#spectrogram label { display: none !important; }
#right-col .chip-val { font-size: 18px; padding: 6px 10px; }
#right-col .chip-row { grid-template-columns: 84px 1fr; margin-bottom: 6px; }


In [ ]:
%%writefile /kaggle/working/rider-voice-space/requirements.txt
# Space runtime. Gradio itself comes from sdk_version in README.md.
# ZeroGPU images already provide torch and the `spaces` package.
torch
transformers==5.17.0
accelerate
safetensors
huggingface_hub
librosa
soundfile
scipy
numpy
torchvision
matplotlib
edge-tts  # optional spoken reply; app falls back to text if missing or offline


In [ ]:
%%writefile /kaggle/working/rider-voice-space/thresholds.json
{
 "safety_tau": 0.02,
 "low_conf": 0.6,
 "sos_tau": 0.42,
 "checkin_tau": 0.04,
 "checkin_timeout_s": 30,
 "source": "round-3 two-level operating point (sos_tau 0.42, checkin_tau 0.04, safety_tau 0.02). Fit on round-3 out-of-fold clips; ship this file with the m1v3 head."
}


In [ ]:
%%writefile /kaggle/working/rider-voice-space/threshold_curve.json
{
  "source": "round-2 held-out synthetic test set (pred_test.jsonl from the balanced-weight head): 620 clips, 31 scripts. The only per-clip predictions on this machine; not the round-3 test set.",
  "gold": "hard = gold action sos (should get SOS); safety = gold safety yes (should get SOS or a check-in); normal = gold safety no",
  "rule": "src/decide.py two-level: SOS if p(action=sos) >= sos_tau; check-in if max(p(safety), p(safety situation), p(sos)) >= checkin_tau",
  "checkin_tau": 0.04,
  "n_clips": 620,
  "n_scripts": 31,
  "n_normal": 440,
  "n_safety": 180,
  "n_hard": 160,
  "curve": [
    {
      "sos_tau": 0.01,
      "false_sos": 0.3431818181818182,
      "safety_recall": 1.0,
      "sos_recall_hard": 1.0,
      "checkin_on_normal": 0.038636363636363635,
      "any_alert_on_normal": 0.38181818181818183,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 151,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 160,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.02,
      "false_sos": 0.275,
      "safety_recall": 1.0,
      "sos_recall_hard": 1.0,
      "checkin_on_normal": 0.09545454545454546,
      "any_alert_on_normal": 0.3704545454545455,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 121,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 160,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.03,
      "false_sos": 0.24545454545454545,
      "safety_recall": 1.0,
      "sos_recall_hard": 1.0,
      "checkin_on_normal": 0.12272727272727273,
      "any_alert_on_normal": 0.36818181818181817,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 108,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 160,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.04,
      "false_sos": 0.21136363636363636,
      "safety_recall": 1.0,
      "sos_recall_hard": 1.0,
      "checkin_on_normal": 0.15454545454545454,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 93,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 160,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.05,
      "false_sos": 0.2,
      "safety_recall": 1.0,
      "sos_recall_hard": 1.0,
      "checkin_on_normal": 0.16590909090909092,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 88,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 160,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.06,
      "false_sos": 0.18409090909090908,
      "safety_recall": 1.0,
      "sos_recall_hard": 1.0,
      "checkin_on_normal": 0.18181818181818182,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 81,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 160,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.07,
      "false_sos": 0.17272727272727273,
      "safety_recall": 1.0,
      "sos_recall_hard": 1.0,
      "checkin_on_normal": 0.19318181818181818,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 76,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 160,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.08,
      "false_sos": 0.16136363636363638,
      "safety_recall": 1.0,
      "sos_recall_hard": 1.0,
      "checkin_on_normal": 0.20454545454545456,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 71,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 160,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.09,
      "false_sos": 0.15681818181818183,
      "safety_recall": 1.0,
      "sos_recall_hard": 1.0,
      "checkin_on_normal": 0.20909090909090908,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 69,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 160,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.1,
      "false_sos": 0.14772727272727273,
      "safety_recall": 1.0,
      "sos_recall_hard": 1.0,
      "checkin_on_normal": 0.21818181818181817,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 65,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 160,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.11,
      "false_sos": 0.1340909090909091,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.9875,
      "checkin_on_normal": 0.2318181818181818,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 59,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 158,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.12,
      "false_sos": 0.12954545454545455,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.9875,
      "checkin_on_normal": 0.23636363636363636,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 57,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 158,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.13,
      "false_sos": 0.125,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.9875,
      "checkin_on_normal": 0.2409090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 55,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 158,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.14,
      "false_sos": 0.12045454545454545,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.9875,
      "checkin_on_normal": 0.24545454545454545,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 53,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 158,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.15,
      "false_sos": 0.1159090909090909,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.98125,
      "checkin_on_normal": 0.25,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 51,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 157,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.16,
      "false_sos": 0.10681818181818181,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.9625,
      "checkin_on_normal": 0.2590909090909091,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 47,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 154,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.17,
      "false_sos": 0.10227272727272728,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.94375,
      "checkin_on_normal": 0.2636363636363636,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 45,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 151,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.18,
      "false_sos": 0.09545454545454546,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.9375,
      "checkin_on_normal": 0.27045454545454545,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 42,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 150,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.19,
      "false_sos": 0.08863636363636364,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.93125,
      "checkin_on_normal": 0.2772727272727273,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 39,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 149,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.2,
      "false_sos": 0.08409090909090909,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.925,
      "checkin_on_normal": 0.2818181818181818,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 37,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 148,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.21,
      "false_sos": 0.08409090909090909,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.925,
      "checkin_on_normal": 0.2818181818181818,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 37,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 148,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.22,
      "false_sos": 0.07954545454545454,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.925,
      "checkin_on_normal": 0.2863636363636364,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 35,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 148,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.23,
      "false_sos": 0.07045454545454545,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.91875,
      "checkin_on_normal": 0.29545454545454547,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 31,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 147,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.24,
      "false_sos": 0.07045454545454545,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.90625,
      "checkin_on_normal": 0.29545454545454547,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 31,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 145,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.25,
      "false_sos": 0.07045454545454545,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.9,
      "checkin_on_normal": 0.29545454545454547,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 31,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 144,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.26,
      "false_sos": 0.0659090909090909,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.9,
      "checkin_on_normal": 0.3,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 29,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 144,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.27,
      "false_sos": 0.06136363636363636,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.8875,
      "checkin_on_normal": 0.30454545454545456,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 27,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 142,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.28,
      "false_sos": 0.05909090909090909,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.88125,
      "checkin_on_normal": 0.3068181818181818,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 26,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 141,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.29,
      "false_sos": 0.05,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.875,
      "checkin_on_normal": 0.3159090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 22,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 140,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.3,
      "false_sos": 0.04772727272727273,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.875,
      "checkin_on_normal": 0.3181818181818182,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 21,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 140,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.31,
      "false_sos": 0.04772727272727273,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.875,
      "checkin_on_normal": 0.3181818181818182,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 21,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 140,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.32,
      "false_sos": 0.045454545454545456,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.875,
      "checkin_on_normal": 0.32045454545454544,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 20,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 140,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.33,
      "false_sos": 0.04318181818181818,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.86875,
      "checkin_on_normal": 0.32272727272727275,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 19,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 139,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.34,
      "false_sos": 0.04090909090909091,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.85625,
      "checkin_on_normal": 0.325,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 18,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 137,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.35,
      "false_sos": 0.04090909090909091,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.85625,
      "checkin_on_normal": 0.325,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 18,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 137,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.36,
      "false_sos": 0.038636363636363635,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.84375,
      "checkin_on_normal": 0.32727272727272727,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 17,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 135,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.37,
      "false_sos": 0.03636363636363636,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.8375,
      "checkin_on_normal": 0.32954545454545453,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 16,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 134,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.38,
      "false_sos": 0.029545454545454545,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.81875,
      "checkin_on_normal": 0.33636363636363636,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 13,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 131,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.39,
      "false_sos": 0.025,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.81875,
      "checkin_on_normal": 0.3409090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 11,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 131,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.4,
      "false_sos": 0.022727272727272728,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.80625,
      "checkin_on_normal": 0.3431818181818182,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 10,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 129,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.41,
      "false_sos": 0.013636363636363636,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.80625,
      "checkin_on_normal": 0.3522727272727273,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 6,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 129,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.42,
      "false_sos": 0.013636363636363636,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.8,
      "checkin_on_normal": 0.3522727272727273,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 6,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 128,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.43,
      "false_sos": 0.013636363636363636,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.8,
      "checkin_on_normal": 0.3522727272727273,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 6,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 128,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.44,
      "false_sos": 0.004545454545454545,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.79375,
      "checkin_on_normal": 0.3613636363636364,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 2,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 127,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.45,
      "false_sos": 0.004545454545454545,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.78125,
      "checkin_on_normal": 0.3613636363636364,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 2,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 125,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.46,
      "false_sos": 0.004545454545454545,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.78125,
      "checkin_on_normal": 0.3613636363636364,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 2,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 125,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.47,
      "false_sos": 0.004545454545454545,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.78125,
      "checkin_on_normal": 0.3613636363636364,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 2,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 125,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.48,
      "false_sos": 0.0022727272727272726,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.775,
      "checkin_on_normal": 0.36363636363636365,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 1,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 124,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.49,
      "false_sos": 0.0022727272727272726,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.76875,
      "checkin_on_normal": 0.36363636363636365,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 1,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 123,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.5,
      "false_sos": 0.0022727272727272726,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.74375,
      "checkin_on_normal": 0.36363636363636365,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 1,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 119,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.51,
      "false_sos": 0.0022727272727272726,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.725,
      "checkin_on_normal": 0.36363636363636365,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 1,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 116,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.52,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.725,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 116,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.53,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.725,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 116,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.54,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.725,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 116,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.55,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.71875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 115,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.56,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.7,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 112,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.57,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.6875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 110,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.58,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.68125,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 109,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.59,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.66875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 107,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.6,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.6625,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 106,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.61,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.65625,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 105,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.62,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.65,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 104,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.63,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.6375,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 102,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.64,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.625,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 100,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.65,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.625,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 100,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.66,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.61875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 99,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.67,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.6125,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 98,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.68,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.6125,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 98,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.69,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.60625,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 97,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.7,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.6,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 96,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.71,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.59375,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 95,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.72,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.59375,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 95,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.73,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.5875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 94,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.74,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.5875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 94,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.75,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.5875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 94,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.76,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.58125,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 93,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.77,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.575,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 92,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.78,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.56875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 91,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.79,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.55625,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 89,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.8,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.55,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 88,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.81,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.5375,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 86,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.82,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.5375,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 86,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.83,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.525,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 84,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.84,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.51875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 83,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.85,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.51875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 83,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.86,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.4875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 78,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.87,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.475,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 76,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.88,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.425,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 68,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.89,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.4125,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 66,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.9,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.4,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 64,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.91,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.38125,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 61,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.92,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.3375,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 54,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.93,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.2875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 46,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.94,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.2125,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 34,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.95,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.14375,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 23,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.96,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.11875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 19,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.97,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.09375,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 15,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.98,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.075,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 12,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    },
    {
      "sos_tau": 0.99,
      "false_sos": 0.0,
      "safety_recall": 1.0,
      "sos_recall_hard": 0.01875,
      "checkin_on_normal": 0.3659090909090909,
      "any_alert_on_normal": 0.3659090909090909,
      "safety_recall_script": 1.0,
      "safety_scripts_fully_caught": 1.0,
      "n_false_sos": 0,
      "n_normal": 440,
      "n_safety_alerted": 180,
      "n_safety": 180,
      "n_hard_sos": 3,
      "n_hard": 160,
      "n_safety_scripts_any": 9,
      "n_safety_scripts_full": 9,
      "n_safety_scripts": 9
    }
  ],
  "caption": "Held-out synthetic test \u00b7 620 clips \u00b7 31 scripts \u00b7 round-2 balanced head. Not the round-3 test set."
}


In [ ]:
%%writefile /kaggle/working/rider-voice-space/spec/decision_set.json
{
 "version": "0.4.0",
 "state": "A delivery rider in India sent this voice note during an order.",
 "questions": {
  "situation": {
   "type": "choice",
   "instructions": "What situation is the rider in?",
   "criteria": {
    "customer_unreachable": "At the drop point, the customer is not answering, not at the door, or the gate is locked",
    "wrong_address": "The address or location is wrong or incomplete, or the rider cannot find the place",
    "restaurant_delay": "At pickup, the order is not ready and the rider is kept waiting",
    "order_issue_at_pickup": "At pickup, items are wrong, missing or unavailable, or the restaurant cancels",
    "food_spilled_or_damaged": "The food has spilled or the package is damaged",
    "vehicle_breakdown": "The rider's vehicle has broken down (puncture, battery, fuel) and the rider is not hurt",
    "road_blocked_or_weather": "Traffic, a closed road, waterlogging or heavy rain is stopping the rider",
    "payment_issue": "A money problem at delivery: no cash or change, UPI failing, or the customer says it is already paid",
    "customer_dispute": "The customer refuses or argues about the order itself (does not want it, wants extras, complains about delay or food), without threatening anyone",
    "accident_or_injury": "The rider has had an accident or someone is injured",
    "harassment_or_threat": "Someone is abusing, threatening or following the rider, or the place feels unsafe",
    "other_or_unclear": "Something else, or the message is too unclear to tell"
   }
  },
  "action": {
   "type": "choice",
   "instructions": "What should the rider do next?",
   "criteria": {
    "wait": "Wait a few more minutes where they are",
    "call_customer": "Call the customer",
    "contact_support": "Contact rider support",
    "request_reassign": "Ask for the order to be reassigned to another rider",
    "return_to_restaurant": "Go back to the restaurant",
    "sos": "Trigger SOS and get urgent human help"
   }
  },
  "safety": {
   "type": "noul",
   "instructions": "Is the rider or someone with them in danger, or does a person need urgent help right now?"
  },
  "language": {
   "type": "choice",
   "instructions": "Which language is the rider speaking?",
   "criteria": {
    "hinglish": "Hinglish (Hindi mixed with English)",
    "hindi": "Hindi",
    "english": "English"
   }
  }
 },
 "action_rules": {
  "customer_unreachable": {
   "default": "call_customer",
   "overrides": [
    {
     "if": {
      "attempts_gte": 2
     },
     "then": "contact_support"
    },
    {
     "if": {
      "waited_min_gte": 10
     },
     "then": "contact_support"
    },
    {
     "if": {
      "rider_waiting": true,
      "waited_min_lt": 5
     },
     "then": "wait"
    }
   ],
   "when": "call_customer while the rider has called at most once and waited under 10 min; contact_support after 2+ calls or 10+ min; wait when the customer has answered and said they are coming (under 5 min so far)."
  },
  "wrong_address": {
   "default": "call_customer",
   "overrides": [
    {
     "if": {
      "attempts_gte": 2
     },
     "then": "contact_support"
    }
   ],
   "when": "call_customer to confirm the address; contact_support once 2+ calls get no answer."
  },
  "restaurant_delay": {
   "default": "wait",
   "overrides": [
    {
     "if": {
      "waited_min_gte": 20
     },
     "then": "request_reassign"
    },
    {
     "if": {
      "not_started": true
     },
     "then": "request_reassign"
    }
   ],
   "when": "wait while the delay is under 20 min; request_reassign at 20+ min or when cooking has not started."
  },
  "order_issue_at_pickup": {
   "default": "contact_support",
   "overrides": [
    {
     "if": {
      "still_at_restaurant": true,
      "fixable": true
     },
     "then": "wait"
    },
    {
     "if": {
      "left_restaurant": true,
      "near_restaurant": true
     },
     "then": "return_to_restaurant"
    }
   ],
   "when": "wait if still at the restaurant and it can be fixed; return_to_restaurant if already left but nearby; otherwise contact_support."
  },
  "food_spilled_or_damaged": {
   "default": "contact_support",
   "overrides": [
    {
     "if": {
      "near_restaurant": true
     },
     "then": "return_to_restaurant"
    }
   ],
   "when": "return_to_restaurant when still near it; otherwise contact_support."
  },
  "vehicle_breakdown": {
   "default": "request_reassign",
   "overrides": [
    {
     "if": {
      "fix_minutes_lte": 10
     },
     "then": "contact_support"
    }
   ],
   "when": "request_reassign; contact_support (keep the order) when the fix takes 10 min or less."
  },
  "road_blocked_or_weather": {
   "default": "wait",
   "overrides": [
    {
     "if": {
      "cannot_proceed": true
     },
     "then": "contact_support"
    }
   ],
   "when": "wait; contact_support when the rider cannot proceed at all."
  },
  "payment_issue": {
   "default": "contact_support",
   "overrides": [
    {
     "if": {
      "customer_fetching_money": true
     },
     "then": "wait"
    }
   ],
   "when": "contact_support; wait while the customer is fetching money."
  },
  "customer_dispute": {
   "default": "contact_support",
   "overrides": [],
   "when": "contact_support - never argue, never return the order without support."
  },
  "accident_or_injury": {
   "default": "sos",
   "overrides": [
    {
     "if": {
      "minor": true,
      "rider_ok": true
     },
     "then": "contact_support"
    }
   ],
   "when": "sos; contact_support only for a minor fall where the rider says they are fine (safety stays yes)."
  },
  "harassment_or_threat": {
   "default": "sos",
   "overrides": [
    {
     "if": {
      "rider_left_safe": true
     },
     "then": "contact_support"
    }
   ],
   "when": "sos - including implicit threats (being followed, cornered, a crowd gathering, keys or phone grabbed); contact_support only once the rider has left and is safe (safety stays yes)."
  },
  "other_or_unclear": {
   "default": "contact_support",
   "overrides": [
    {
     "if": {
      "medical_emergency": true
     },
     "then": "sos"
    }
   ],
   "when": "contact_support; sos for a medical emergency (chest pain, fainting, can't breathe, heat collapse)."
  }
 },
 "safety_rules": {
  "always_yes_situations": [
   "accident_or_injury",
   "harassment_or_threat"
  ],
  "yes_if_cue": [
   "medical_emergency",
   "dangerous_place"
  ]
 },
 "language_rule": {
  "loanwords_counted_as_hindi": [
   "order",
   "phone",
   "call",
   "address",
   "location",
   "gate",
   "customer",
   "restaurant",
   "bike",
   "otp",
   "app",
   "support",
   "delivery",
   "packet",
   "cancel",
   "payment",
   "upi",
   "cash",
   "minute",
   "building",
   "flat",
   "society",
   "lift",
   "signal"
  ],
  "hinglish_min_other_english_words": 2
 },
 "scopes": {
  "m1": {
   "description": "door-step wedge: what can happen at or on the way to the customer's door",
   "situations": [
    "customer_unreachable",
    "wrong_address",
    "payment_issue",
    "customer_dispute",
    "accident_or_injury",
    "harassment_or_threat",
    "other_or_unclear"
   ]
  }
 },
 "situation_boundaries": [
  "payment_issue vs customer_dispute: payment is about the money (cash, change, UPI, 'already paid'); dispute is about the order itself (refuses it, wants extras, angry about delay or food). 'Refuses to pay because it is prepaid' is payment.",
  "customer_dispute vs harassment_or_threat: shouting or complaining is dispute; any threat of harm, blocking the rider's way, grabbing belongings or a crowd closing in is harassment.",
  "accident_or_injury vs vehicle_breakdown: if the rider or anyone is hurt it is accident_or_injury; a fallen bike with nobody hurt is vehicle_breakdown.",
  "someone else's accident blocking the road is road_blocked_or_weather, safety no."
 ],
 "alerts": {
  "levels": [
   "sos",
   "checkin",
   "none"
  ],
  "sos": "p(action = sos) >= sos_tau -> emergency card + human escalation",
  "checkin": "max(p(safety = yes), p(safety situation), p(sos)) >= checkin_tau -> 'Are you safe?' card; escalates to SOS if unanswered for checkin_timeout_s",
  "gold": "hard = gold action sos (should get SOS); soft = other safety=yes, e.g. a minor fall where the rider is fine or the rider has already left a threat (a check-in is enough); none = safety no"
 }
}

In [ ]:
%%writefile /kaggle/working/rider-voice-space/src/__init__.py


In [ ]:
%%writefile /kaggle/working/rider-voice-space/src/alert_rules.py
"""Torch-free copy of the published two-level alert rule and product card.

The rule is defined in src/decide.py (safety_scores, alert_level, consistent) and src/product.py
(card). This module exists so the demo UI, the threshold slider, and
scripts/build_threshold_curve.py can apply that same rule without importing torch. Do not change
the rule here — Claude owns the logic in decide.py / product.py.
"""

from __future__ import annotations

SAFETY_SITUATIONS = {"accident_or_injury", "harassment_or_threat"}
SAFE_ACTIONS = ("sos", "contact_support")
DEFAULTS = {"safety_tau": 0.2, "low_conf": 0.6, "checkin_timeout_s": 30}


def safety_scores(ans: dict) -> dict:
    """The three safety signals and the combined check-in score, from answers()."""
    sit = ans["situation"]["probabilities"]
    p_sos = ans["action"]["probabilities"].get("sos", 0.0)
    p_safe = ans["safety"]["noul"]
    p_sit = sum(sit.get(s, 0.0) for s in SAFETY_SITUATIONS)
    return {"p_safety": p_safe, "p_sos": p_sos, "p_safety_situation": p_sit, "checkin_score": max(p_safe, p_sit, p_sos)}


def alert_level(ans: dict, th: dict) -> str:
    """'sos' | 'checkin' | 'none'. Without two-level thresholds: legacy single flag (sos when p(safety) >= safety_tau
    or a safety situation is top-1)."""
    sc = safety_scores(ans)
    if "sos_tau" in th and "checkin_tau" in th:
        if sc["p_sos"] >= th["sos_tau"]:
            return "sos"
        return "checkin" if sc["checkin_score"] >= th["checkin_tau"] else "none"
    tau = th.get("safety_tau", 0.2)
    if sc["p_safety"] >= tau or ans["situation"]["choice"] in SAFETY_SITUATIONS or ans["action"]["choice"] == "sos":
        return "sos"
    return "none"


def consistent(ans: dict, th) -> tuple[dict, dict]:
    """At the SOS level the action is restricted to {sos, contact_support}.
    th: thresholds dict (or, for old callers, a float safety_tau).
    Returns (answers with ans["action"]["choice"] possibly changed, report)."""
    th = {"safety_tau": th} if isinstance(th, (int, float)) else th
    level = alert_level(ans, th)
    flag = level == "sos"
    sit_safe = ans["situation"]["choice"] in SAFETY_SITUATIONS
    raw = ans["action"]["choice"]
    rep = {"level": level, "safety_flag": flag, "safety_situation": sit_safe, "raw_action": raw, "overridden": False,
           "disagree": flag and raw not in SAFE_ACTIONS, **safety_scores(ans)}
    if rep["disagree"]:
        p = ans["action"]["probabilities"]
        new = max(SAFE_ACTIONS, key=lambda a: p.get(a, 0.0))
        ans = {**ans, "action": {**ans["action"], "choice": new}}
        rep["overridden"] = True
    return ans, rep


def _suggestion(answers: dict, th: dict) -> dict:
    act_p = answers["action"]["probabilities"]
    ranked = sorted((a for a in act_p if act_p[a] > 0), key=act_p.get, reverse=True) or sorted(act_p, key=act_p.get, reverse=True)
    sit = answers["situation"]["choice"]
    support = [] if "contact_support" in ranked[:2] else ["contact_support"]
    low = act_p[ranked[0]] < th["low_conf"]
    if low or sit == "other_or_unclear":
        return {"kind": "choose", "primary": None, "alternatives": ranked[:2] + support,
                "reasons": ["low confidence" if low else "unclear situation"]}
    return {"kind": "suggest", "primary": answers["action"]["choice"], "alternatives": [a for a in ranked if a != answers["action"]["choice"]][:1] + support,
            "reasons": []}


def card(answers: dict, thresholds: dict | None = None) -> dict:
    th = {**DEFAULTS, **(thresholds or {})}
    level = alert_level(answers, th)
    sc = {k: round(v, 4) for k, v in safety_scores(answers).items()}
    if level == "sos":
        return {"kind": "sos", "primary": "sos", "alternatives": ["contact_support"], "escalate_to_human": True,
                "reasons": [f"p(sos)={sc['p_sos']:.2f}" if "sos_tau" in th else "safety flag"], "scores": sc,
                "requires_tap": True}
    sug = _suggestion(answers, th)
    if level == "checkin":
        return {"kind": "checkin", "question": "Are you safe?", "buttons": ["i_am_safe", "get_help"],
                "auto_escalate_after_s": th["checkin_timeout_s"], "escalate_to_human": False,
                "then": sug, "primary": sug["primary"], "alternatives": sug["alternatives"],
                "reasons": [f"safety score {sc['checkin_score']:.2f} >= {th['checkin_tau']:.2f}"], "scores": sc,
                "requires_tap": True}
    return {**sug, "escalate_to_human": False, "scores": sc, "requires_tap": True}


In [ ]:
%%writefile /kaggle/working/rider-voice-space/src/backbone.py
"""The frozen backbone: one forward pass per clip -> pooled features for every question.

Mirrors `NirnayaAudioEngine._prepare/_finish` from nirnaya-inference (mmclm/di_engine.py) without vLLM, so it runs on
Kaggle T4s (no bf16) and on CPU:

  audio first, then the joint prompt text (mmclm.joint) written twice when short enough; the head reads the spans of
  the second copy (mmclm.jh.joint_spans) from the final, post-norm hidden states; mmclm.jh.pool_joint pools them.

Precision modes
  fp32  (default on T4): weights loaded as fp16 onto the GPUs, then every module except the per-layer embedding
        table is upcast to fp32 in place. The per-layer table (~2.8 B params) is a lookup and stays fp16
        (exact for the trained bf16 values in fp16's normal range). ~26 GB across 2x T4.
  fp16  everything fp16 (~16 GB). Faster; Gemma activations can overflow in fp16 - `features()` checks for inf/nan.
  bf16  for GPUs that support it (A100/L4/RTX Pro 6000) - the closest to how the model was trained.
  On CPU, fp32 only.
"""

from __future__ import annotations

import io
import math
from dataclasses import dataclass, field

import numpy as np
import torch

THOUGHT = "<|channel>thought\n<channel|>"  # the empty thought add_generation_prompt appends with thinking off
SAMPLE_RATE = 16000
PLE = "model.language_model.embed_tokens_per_layer"


def _mmclm():
    try:
        from mmclm import jh, joint
    except ImportError as e:  # pragma: no cover
        raise ImportError("nirnaya-inference is not importable. Clone github.com/NeverInAsh/nirnaya-inference "
                          "(pinned commit in requirements.txt) and add it to PYTHONPATH.") from e
    return jh, joint


def load_audio(a, sr: int = SAMPLE_RATE) -> np.ndarray:
    """Path, bytes or array -> float32 mono at 16 kHz (same as the reference engine: librosa, mono)."""
    if isinstance(a, np.ndarray):
        return a.astype(np.float32)
    import librosa

    src = io.BytesIO(a) if isinstance(a, (bytes, bytearray)) else a
    return librosa.load(src, sr=sr, mono=True)[0].astype(np.float32)


# --------------------------------------------------------------------------- device planning
def _module_sizes(config) -> dict[str, int]:
    """Parameter count of every placement unit, from a meta-device model (no weights loaded)."""
    from transformers import Gemma4ForConditionalGeneration

    with torch.device("meta"):
        m = Gemma4ForConditionalGeneration(config)
    sizes: dict[str, int] = {}
    for name, p in m.named_parameters():
        parts = name.split(".")
        unit = ".".join(parts[:4]) if name.startswith("model.language_model.layers.") else ".".join(parts[:3])
        if name.startswith("lm_head"):
            unit = "lm_head"
        sizes[unit] = sizes.get(unit, 0) + p.numel()
    return sizes


def plan_device_map(config, n_gpus: int, gpu_bytes: list[int], precision: str, reserve_gb: float = 1.6) -> dict:
    """Embeddings, the per-layer table and the audio tower on cuda:0; decoder layers filled in order; final norm on the
    last GPU used. Sizes are counted at the *final* precision (fp32 except the per-layer table in fp32 mode)."""
    sizes = _module_sizes(config)
    bpp = {"fp32": 4, "fp16": 2, "bf16": 2}[precision]

    def nbytes(unit):
        return sizes[unit] * (2 if unit == PLE else bpp)

    budget = [b - int(reserve_gb * 2**30) for b in gpu_bytes[:n_gpus]]
    dmap, used, dev = {}, [0] * n_gpus, 0
    first = [u for u in sizes if not u.startswith("model.language_model.layers.") and u not in ("model.language_model.norm",)]
    for u in first:
        dmap[u], used[0] = 0, used[0] + nbytes(u)
    layers = sorted((u for u in sizes if u.startswith("model.language_model.layers.")), key=lambda u: int(u.split(".")[-1]))
    for u in layers:
        while dev < n_gpus - 1 and used[dev] + nbytes(u) > budget[dev]:
            dev += 1
        dmap[u], used[dev] = dev, used[dev] + nbytes(u)
    dmap["model.language_model.norm"] = dev
    dmap["model.language_model.rotary_emb"] = 0
    dmap["lm_head"] = 0  # tied to embed_tokens; never called
    if any(used[i] > budget[i] for i in range(n_gpus)):
        gb = [round(x / 2**30, 1) for x in used]
        raise RuntimeError(f"model does not fit: planned {gb} GB vs budgets {[round(b / 2**30, 1) for b in budget]}; "
                           f"try precision='fp16' or more GPUs")
    return dmap


# --------------------------------------------------------------------------- backbone
@dataclass
class Prepared:
    inputs: dict
    qs: list            # [(qid, kind, keys, jq)]
    shown: list         # [(jq, perm)]
    twice: bool
    n_tokens: int = 0
    extra: dict = field(default_factory=dict)


class Backbone:
    def __init__(self, model_dir: str, precision: str | None = None, device: str | None = None, twice_max: int = 3000,
                 mem_cap: int = 256, max_audio_s: float = 30.0, attn: str = "sdpa", verbose: bool = True,
                 state_variant: str = "post_norm", placement: str = "auto"):
        """placement: auto = plan a split over every visible GPU (Kaggle 2x T4); single = whole model on one device
        with .to(device) (one big GPU, e.g. a Hugging Face ZeroGPU Space)."""
        from transformers import AutoConfig, AutoProcessor, Gemma4ForConditionalGeneration

        self.jh, self.joint = _mmclm()
        self.twice_max, self.mem_cap, self.max_audio_s = int(twice_max), int(mem_cap), float(max_audio_s)
        if state_variant not in ("post_norm", "pre_norm"):
            raise ValueError(state_variant)
        self.state_variant = state_variant  # post_norm = what vLLM's token_embed pooler returns (the reference)
        cuda = torch.cuda.is_available() and device != "cpu"
        if precision is None:
            native_bf16 = cuda and torch.cuda.get_device_capability(0)[0] >= 8  # T4 (7.5) only emulates bf16
            precision = "bf16" if native_bf16 else "fp32"
        self.precision = precision
        if not cuda and self.precision != "fp32":
            raise ValueError("on CPU use precision='fp32'")

        self.proc = AutoProcessor.from_pretrained(model_dir)
        self.tok = self.proc.tokenizer
        cfg = AutoConfig.from_pretrained(model_dir)
        cfg.vision_config = None  # audio + text only: skip loading the vision tower
        load_dtype = {"fp32": torch.float16 if cuda else torch.float32, "fp16": torch.float16, "bf16": torch.bfloat16}[self.precision]
        kw = dict(config=cfg, dtype=load_dtype, attn_implementation=attn)
        if cuda and placement == "auto":
            n = torch.cuda.device_count()
            mem = [torch.cuda.get_device_properties(i).total_memory for i in range(n)]
            kw["device_map"] = plan_device_map(cfg, n, mem, self.precision)
        model = Gemma4ForConditionalGeneration.from_pretrained(model_dir, **kw)
        if cuda and placement == "single":
            if self.precision == "fp32":
                raise ValueError("placement='single' expects fp16/bf16 (the fp32 path is for splitting over T4s)")
            model = model.to("cuda")
        if self.precision == "fp32" and cuda:
            for name, p in model.named_parameters():  # upcast in place, module by module, leaving the PLE table fp16
                if not name.startswith(PLE) and p.dtype != torch.float32:
                    p.data = p.data.float()
            for mod in model.modules():  # rotary / positional buffers too
                for bn, b in list(mod._buffers.items()):
                    if b is not None and b.is_floating_point() and b.dtype != torch.float32:
                        mod._buffers[bn] = b.float()
            torch.cuda.empty_cache()
        self.model = model.eval()
        self.hidden = cfg.text_config.hidden_size
        self.in_device = next(model.model.language_model.embed_tokens.parameters()).device
        self.feat_dtype = torch.float32 if self.precision == "fp32" else load_dtype
        if verbose:
            devs = sorted({str(p.device) for p in model.parameters()})
            print(f"[backbone] precision={self.precision} devices={devs} hidden={self.hidden} "
                  f"mem_cap={self.mem_cap} twice_max={self.twice_max}")

    # ----------------------------------------------------------------- prompt (mirrors NirnayaAudioEngine._prepare)
    def prepare(self, state, questions: dict, audio=None) -> Prepared:
        joint = self.joint
        qs = []
        for qk, q in questions.items():
            kind = q.get("type", "choice")
            if kind == "noul":
                keys, opts = ["true", "false"], ["yes", "no"]
            else:
                keys = list(q.get("criteria") or {})
                if not 2 <= len(keys) <= joint.MAX_OPTIONS:
                    raise ValueError(f"{qk}: {len(keys)} options")
                opts = [str(q["criteria"][k]).strip() or k for k in keys]
            qs.append((qk, kind, keys, {"instruction": str(q.get("instructions", "")).strip(), "type": kind,
                                         "options": opts, "gold": 0}))
        shown = [(jq, list(range(len(jq["options"])))) for *_, jq in qs]
        head_text = "" if state in ("", None, {}, []) else str(state).strip()
        text = joint.prompt_text(head_text, shown)
        twice = bool(self.twice_max) and len(self.tok(text, add_special_tokens=False)["input_ids"]) <= self.twice_max
        wav = None if audio is None else load_audio(audio)
        if wav is not None and len(wav) > self.max_audio_s * SAMPLE_RATE:
            wav = wav[: int(self.max_audio_s * SAMPLE_RATE)]
        content = ([{"type": "audio"}] if wav is not None else []) + [{"type": "text", "text": f"{text}\n\n{text}" if twice else text}]
        pr = self.proc.apply_chat_template([{"role": "user", "content": content}], tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
        if twice and pr.endswith(THOUGHT):
            pr = pr[: -len(THOUGHT)]
        # the chat template already writes <bos>; don't let the tokenizer add a second one
        if wav is not None:
            inputs = self.proc(text=pr, audio=[wav], return_tensors="pt", add_special_tokens=False)
        else:
            inputs = self.proc(text=pr, return_tensors="pt", add_special_tokens=False)
        ids = inputs["input_ids"][0]
        if int((ids == self.tok.bos_token_id).sum()) != 1:
            raise RuntimeError(f"expected exactly one <bos>, got {int((ids == self.tok.bos_token_id).sum())}")
        return Prepared(inputs=dict(inputs), qs=qs, shown=shown, twice=twice, n_tokens=len(ids),
                        extra={"audio_s": 0.0 if wav is None else len(wav) / SAMPLE_RATE})

    def spans(self, ids: list[int], p: Prepared) -> dict:
        jh, joint = self.jh, self.joint
        off = 0
        if p.twice:  # the second copy starts after the first copy's closing instruction
            instr = self.tok(joint.INSTRUCTION, add_special_tokens=False)["input_ids"]
            j = jh._find(ids, instr, 0)
            if j < 0:
                raise RuntimeError("closing instruction of the first copy not found")
            off = j + len(instr)
        sp = jh.joint_spans(self.tok, ids[off:], p.shown)
        if sp is None:
            raise RuntimeError("joint layout spans not found")
        if off:
            sh = lambda ab: (ab[0] + off, ab[1] + off)  # noqa: E731
            sp = {"state": sh(sp["state"]), "questions": [sh(x) for x in sp["questions"]],
                  "options": [[sh(x) for x in o] for o in sp["options"]]}
        return sp

    # ----------------------------------------------------------------- forward
    @torch.no_grad()
    def hidden_states(self, p: Prepared) -> torch.Tensor:
        """[T, H] final (post-norm) hidden states of the prompt, the same states vLLM's token_embed pooler returns."""
        x = {k: v.to(self.in_device) for k, v in p.inputs.items() if torch.is_tensor(v)}
        if "input_features" in x:
            x["input_features"] = x["input_features"].to(self.feat_dtype)
        if self.state_variant == "post_norm":
            return self.model.model(**x, use_cache=False, return_dict=True).last_hidden_state[0]
        # pre_norm: the input of the final RMSNorm (transformers replaces the last hidden_states entry with the
        # normed one, so it is captured with a hook instead)
        grab = {}
        hook = self.model.model.language_model.norm.register_forward_pre_hook(lambda m, args: grab.__setitem__("h", args[0]))
        try:
            self.model.model(**x, use_cache=False, return_dict=True)
        finally:
            hook.remove()
        return grab["h"][0]

    @torch.no_grad()
    def features(self, state, questions: dict, audio=None) -> dict:
        """One clip -> {"memory": [C,H] fp16, "items": [per question {end, question, options}], "meta": {...}}.
        `memory` is shared by every question of the clip (pool_joint gives each question the same token memory)."""
        p = self.prepare(state, questions, audio)
        h = self.hidden_states(p).float()
        if not torch.isfinite(h).all():
            raise FloatingPointError(f"non-finite hidden states (precision={self.precision})")
        ids = p.inputs["input_ids"][0].tolist()
        sp = self.spans(ids, p)
        self.jh.MEM_CAP = self.mem_cap if self.mem_cap > 0 else 2048
        pooled = self.jh.pool_joint(h, sp, len(h) - 1)
        mem = pooled[0]["chunks"]
        items = [{"qid": qk, "kind": kind, "keys": keys, "end": f["end"].half().cpu(), "question": f["question"].half().cpu(),
                  "options": f["options"].half().cpu()} for (qk, kind, keys, _), f in zip(p.qs, pooled)]
        audio_id = getattr(self.tok, "audio_token_id", None)
        meta = {"n_tokens": len(ids), "n_audio_tokens": int(sum(t == audio_id for t in ids)) if audio_id is not None else None,
                "twice": p.twice, "mem_cap": self.mem_cap, "precision": self.precision, "audio_s": p.extra["audio_s"],
                "state_variant": self.state_variant,
                "h_absmax": float(h.abs().max())}
        return {"memory": mem.half().cpu(), "items": items, "meta": meta}


def feature_items_for_head(feat: dict) -> list[dict]:
    """Per-question dicts in the shape mmclm.jh.collate expects (shared memory, zero digit prior)."""
    out = []
    for it in feat["items"]:
        k = it["options"].shape[0]
        out.append({"end": it["end"], "question": it["question"], "options": it["options"], "chunks": feat["memory"],
                    "prior": torch.zeros(k), "exact_prior": False})
    return out


def entropy_temperature(lg: torch.Tensor, fit: dict | None, t_global: float = 1.0) -> float:
    """T = exp(a + b*H + c*H^2 + d*log k), H the normalised entropy of softmax(lg) (as in di_engine._entropy_t)."""
    if not fit:
        return t_global
    a, b, c, d = (float(fit.get(x, 0.0)) for x in "abcd")
    k = lg.numel()
    lp = lg.float().log_softmax(-1)
    h = float(-(lp.exp() * lp).sum()) / math.log(k) if k > 1 else 0.0
    return math.exp(a + b * h + c * h * h + d * math.log(k))


In [ ]:
%%writefile /kaggle/working/rider-voice-space/src/decide.py
"""Shared by training, prediction and serving: features -> head logits -> calibrated probabilities -> answers.

Question order and option keys always come from spec/decision_set.json, so labels, logits and probabilities line up.

Scope: a milestone (e.g. "m1", the door-step wedge) only covers some situations. Out-of-scope options are masked to
-inf before the softmax - in training, calibration, prediction and serving alike - so the head cannot spend probability
on labels it never sees. The in-scope actions are derived from the action rules of the in-scope situations.

Two-level safety (round 3):
  SOS       p(action = sos) >= sos_tau                                   -> emergency card, human escalation
  check-in  safety score >= checkin_tau, where safety score =
            max(p(safety = yes), p(situation in {accident, harassment}), p(action = sos))
                                                                          -> "Are you safe?" one-tap card that
                                                                             escalates by itself if unanswered
Both thresholds are fitted on out-of-fold clips (train/calibrate.py). Runs without them (round 1/2) fall back to the
single safety_tau flag.

Consistency rule: when the SOS level fires the action is restricted to {sos, contact_support}. A check-in does not
rewrite the action (round 2 showed that rewriting on every low-threshold flag cost ~6 points of action accuracy).
The raw model action is always kept alongside.
"""

from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from src.labels import options, spec  # noqa: E402

QUESTIONS = list(spec()["questions"])           # situation, action, safety, language
N_OPTIONS = {q: len(options(q)) for q in QUESTIONS}
SAFETY_SITUATIONS = set(spec()["safety_rules"]["always_yes_situations"])
SAFE_ACTIONS = ("sos", "contact_support")


# --------------------------------------------------------------------------- scope
def scope_for(name: str | None) -> dict | None:
    """{"situation": [...], "action": [...]} for a named scope in the spec, or None (= every option)."""
    if not name:
        return None
    sc = spec().get("scopes", {}).get(name)
    if sc is None:
        raise ValueError(f"unknown scope {name!r}; spec has {list(spec().get('scopes', {}))}")
    sits = list(sc["situations"])
    acts = set()
    for s in sits:
        r = spec()["action_rules"][s]
        acts.add(r["default"])
        acts.update(o["then"] for o in r["overrides"])
    return {"situation": [s for s in options("situation") if s in sits], "action": [a for a in options("action") if a in acts]}


def scope_mask(scope: dict | None) -> dict[str, torch.Tensor]:
    """question -> bool tensor (True = allowed)."""
    out = {}
    for q in QUESTIONS:
        keys = options(q)
        allowed = (scope or {}).get(q)
        out[q] = torch.tensor([allowed is None or k in allowed for k in keys])
    return out


def apply_mask(lg: torch.Tensor, allowed: torch.Tensor) -> torch.Tensor:
    return lg.masked_fill(~allowed.to(lg.device), float("-inf"))


# --------------------------------------------------------------------------- head
def gold_index(q: str, row: dict) -> int:
    """Index of the gold option for question q in a manifest/index row (-1 when the label is missing)."""
    v = row.get(q)
    if v is None:
        return -1
    if q == "safety":
        v = "true" if (v is True or str(v).lower() in ("true", "yes", "1")) else "false"
    keys = options(q)
    if v not in keys:
        raise ValueError(f"{row.get('clip_id')}: {q}={v!r} not in {keys}")
    return keys.index(v)


def head_items(rec: dict) -> list[dict]:
    """A cached feature record -> per-question items for mmclm.jh.collate (shared memory, zero digit prior)."""
    by_q = {it["qid"]: it for it in rec["items"]}
    out = []
    for q in QUESTIONS:
        it = by_q[q]
        out.append({"end": it["end"], "question": it["question"], "options": it["options"], "chunks": rec["memory"],
                    "prior": torch.zeros(it["options"].shape[0]), "exact_prior": False})
    return out


def head_logits(head, jh, recs: list[dict], device, scope: dict | None = None) -> list[dict[str, torch.Tensor]]:
    """Raw (T=1) logits per clip per question, on CPU, out-of-scope options at -inf."""
    items = [x for r in recs for x in head_items(r)]
    nq, masks = len(QUESTIONS), scope_mask(scope)
    lg_all = torch.cat([head(jh.collate(items[c0:c0 + 32], device)).float().cpu() for c0 in range(0, len(items), 32)])
    return [{q: apply_mask(lg_all[i * nq + j, : N_OPTIONS[q]], masks[q]) for j, q in enumerate(QUESTIONS)}
            for i in range(len(recs))]


# --------------------------------------------------------------------------- calibration
def temperature(lg: torch.Tensor, fit_q: dict | None) -> float:
    """T = exp(a + b*H + c*H^2 + d*log k) over the finite (in-scope) logits; H = normalised entropy at T=1."""
    if not fit_q:
        return 1.0
    a, b, c, d = (float(fit_q.get(x, 0.0)) for x in "abcd")
    x = lg[torch.isfinite(lg)].float()
    k = x.numel()
    lp = x.log_softmax(-1)
    h = float(-(lp.exp() * lp).sum()) / math.log(k) if k > 1 else 0.0
    return math.exp(a + b * h + c * h * h + d * math.log(max(k, 1)))


def fit_for(fit: dict | None, q: str) -> dict | None:
    """Pooled fit ({"a".."d"}) or per-question fit ({"per_question": {q: {"a".."d"}}})."""
    if not fit:
        return None
    if "per_question" in fit:
        return fit["per_question"].get(q)
    return fit


def calibrated_probs(logits: dict[str, torch.Tensor], fit: dict | None) -> dict[str, list[float]]:
    return {q: (lg / temperature(lg, fit_for(fit, q))).softmax(-1).tolist() for q, lg in logits.items()}


# --------------------------------------------------------------------------- answers + consistency
def answers(probs: dict[str, list[float]]) -> dict:
    """Decision Index style answers: choice -> {choice, probabilities}; yes/no -> {noul: p(yes)}."""
    out = {}
    for q, p in probs.items():
        keys = options(q)
        if spec()["questions"][q]["type"] == "noul":
            out[q] = {"type": "noul", "noul": p[0]}
        else:
            d = dict(zip(keys, p))
            out[q] = {"type": "choice", "choice": max(d, key=d.get), "probabilities": d}
    return out


def safety_scores(ans: dict) -> dict:
    """The three safety signals and the combined check-in score, from answers()."""
    sit = ans["situation"]["probabilities"]
    p_sos = ans["action"]["probabilities"].get("sos", 0.0)
    p_safe = ans["safety"]["noul"]
    p_sit = sum(sit.get(s, 0.0) for s in SAFETY_SITUATIONS)
    return {"p_safety": p_safe, "p_sos": p_sos, "p_safety_situation": p_sit, "checkin_score": max(p_safe, p_sit, p_sos)}


def alert_level(ans: dict, th: dict) -> str:
    """'sos' | 'checkin' | 'none'. Without two-level thresholds: legacy single flag (sos when p(safety) >= safety_tau
    or a safety situation is top-1)."""
    sc = safety_scores(ans)
    if "sos_tau" in th and "checkin_tau" in th:
        if sc["p_sos"] >= th["sos_tau"]:
            return "sos"
        return "checkin" if sc["checkin_score"] >= th["checkin_tau"] else "none"
    tau = th.get("safety_tau", 0.2)
    if sc["p_safety"] >= tau or ans["situation"]["choice"] in SAFETY_SITUATIONS or ans["action"]["choice"] == "sos":
        return "sos"
    return "none"


def consistent(ans: dict, th) -> tuple[dict, dict]:
    """At the SOS level the action is restricted to {sos, contact_support}.
    th: thresholds dict (or, for old callers, a float safety_tau).
    Returns (answers with ans["action"]["choice"] possibly changed, report)."""
    th = {"safety_tau": th} if isinstance(th, (int, float)) else th
    level = alert_level(ans, th)
    flag = level == "sos"
    sit_safe = ans["situation"]["choice"] in SAFETY_SITUATIONS
    raw = ans["action"]["choice"]
    rep = {"level": level, "safety_flag": flag, "safety_situation": sit_safe, "raw_action": raw, "overridden": False,
           "disagree": flag and raw not in SAFE_ACTIONS, **safety_scores(ans)}
    if rep["disagree"]:
        p = ans["action"]["probabilities"]
        new = max(SAFE_ACTIONS, key=lambda a: p.get(a, 0.0))
        ans = {**ans, "action": {**ans["action"], "choice": new}}
        rep["overridden"] = True
    return ans, rep


def load_run(run_dir: str, device="cpu"):
    """head.pt (+ t_entropy.json, thresholds.json, config.json) from a training run directory.
    Returns (head, jh, fit, thresholds, config, scope)."""
    from mmclm import jh

    run = Path(run_dir)
    head = jh.load_head(str(run / "head.pt"), device=device)
    fit = json.loads((run / "t_entropy.json").read_text()) if (run / "t_entropy.json").exists() else None
    th = json.loads((run / "thresholds.json").read_text()) if (run / "thresholds.json").exists() else {}
    cfg = json.loads((run / "config.json").read_text()) if (run / "config.json").exists() else {}
    return head, jh, fit, th, cfg, cfg.get("scope")


In [ ]:
%%writefile /kaggle/working/rider-voice-space/src/engine.py
"""End-to-end: a voice note -> calibrated answers for all four questions -> the one-tap card. For the demo and the
latency benchmark. Uses exactly the extraction settings the head was trained with (read from the run's config).

    python src/engine.py --model <model dir> --run runs/m1 clip.wav [clip2.wav ...] [--bench 100] [--precision fp16]

--precision overrides the precision the features were extracted with (for latency runs; check accuracy with an
fp16 extraction of the test split before serving that way - notebook 02 does both).

Also usable as a Decision Index engine: `RiderVoiceEngine(...)(state, questions, audios=[...])`.
"""

from __future__ import annotations

import argparse
import json
import statistics
import sys
import time
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from src.backbone import Backbone  # noqa: E402
from src.decide import answers, calibrated_probs, consistent, head_logits, load_run  # noqa: E402
from src.product import DEFAULTS, card  # noqa: E402
from src.request import build_request  # noqa: E402


class RiderVoiceEngine:
    name = "rider-voice-decisions"

    def __init__(self, model: str, run: str, device=None, precision=None, placement="auto", thresholds: dict | None = None):
        self.head_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.head, self.jh, self.fit, self.th, cfg, self.scope = load_run(run, self.head_device)
        if thresholds:
            self.th = {**self.th, **thresholds}
        fm = cfg.get("features_meta", {})
        self.bb = Backbone(model, precision=precision or fm.get("precision"), mem_cap=fm.get("mem_cap", 256),
                           twice_max=fm.get("twice_max", 3000), device=None if self.head_device != "cpu" else "cpu",
                           state_variant=fm.get("state_variant", "post_norm"), placement=placement)
        self.state, self.questions = build_request()

    @torch.no_grad()
    def decide(self, audio) -> dict:
        t0 = time.perf_counter()
        f = self.bb.features(self.state, self.questions, audio)
        rec = {"memory": f["memory"], "items": f["items"]}
        lg = head_logits(self.head, self.jh, [rec], self.head_device, self.scope)[0]
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        ans, rep = consistent(answers(calibrated_probs(lg, self.fit)), self.th or {"safety_tau": DEFAULTS["safety_tau"]})
        return {"answers": ans, "consistency": rep, "card": card(ans, self.th),
                "latency_ms": round(1000 * (time.perf_counter() - t0), 1), "precision": self.bb.precision,
                "audio_s": f["meta"]["audio_s"], "n_tokens": f["meta"]["n_tokens"]}

    def __call__(self, state, questions, audios=()):
        """Decision Index-style call for the fixed rider request (state/questions must be this repo's spec)."""
        if not audios:
            raise ValueError("this engine needs one audio clip")
        out = self.decide(audios[0])
        return {"model": self.name, "answers": out["answers"]}, {"latency_ms": out["latency_ms"]}


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--run", required=True)
    ap.add_argument("clips", nargs="+")
    ap.add_argument("--bench", type=int, default=0, help="time N calls (after 5 warm-up) and report p50/p95")
    ap.add_argument("--device", default=None)
    ap.add_argument("--precision", default=None, choices=[None, "fp32", "fp16", "bf16"])
    a = ap.parse_args(argv)
    eng = RiderVoiceEngine(a.model, a.run, a.device, a.precision)
    for c in a.clips:
        out = eng.decide(c)
        print(json.dumps({"clip": c, **out}, indent=1, ensure_ascii=False))
    if a.bench:
        for i in range(5):
            eng.decide(a.clips[i % len(a.clips)])
        ts = [eng.decide(a.clips[i % len(a.clips)])["latency_ms"] for i in range(a.bench)]
        ts.sort()
        print(json.dumps({"n": len(ts), "precision": eng.bb.precision, "p50_ms": statistics.median(ts), "p95_ms": ts[int(0.95 * (len(ts) - 1))],
                          "mean_ms": round(statistics.mean(ts), 1),
                          "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/rider-voice-space/src/labels.py
"""The decision set and the rules that turn a script's structured cues into gold labels.

Gold `action` and `safety` are never hand-picked: they are derived from (situation, cues) by the rules in
spec/decision_set.json, so every label is reproducible and auditable.

    python src/labels.py          # self-test
"""

from __future__ import annotations

import json
import re
from functools import lru_cache
from pathlib import Path

SPEC_PATH = Path(__file__).resolve().parent.parent / "spec" / "decision_set.json"


@lru_cache(maxsize=1)
def spec() -> dict:
    return json.loads(SPEC_PATH.read_text(encoding="utf-8"))


def options(question: str) -> list[str]:
    q = spec()["questions"][question]
    return ["true", "false"] if q["type"] == "noul" else list(q["criteria"])


SITUATIONS = options("situation")
ACTIONS = options("action")
LANGUAGES = options("language")


def _match(cond: dict, cues: dict) -> bool:
    """A condition like {"attempts_gte": 2, "rider_waiting": true} holds when every clause holds."""
    for key, want in cond.items():
        for suffix, op in (("_gte", lambda a, b: a >= b), ("_lte", lambda a, b: a <= b), ("_lt", lambda a, b: a < b)):
            if key.endswith(suffix):
                have = cues.get(key[: -len(suffix)])
                if have is None or not op(have, want):
                    return False
                break
        else:
            if cues.get(key) != want:
                return False
    return True


def derive_action(situation: str, cues: dict | None = None) -> str:
    """First matching override wins, else the situation's default."""
    if situation not in SITUATIONS:
        raise ValueError(f"unknown situation {situation!r}")
    rule, cues = spec()["action_rules"][situation], cues or {}
    for o in rule["overrides"]:
        if _match(o["if"], cues):
            return o["then"]
    return rule["default"]


def derive_safety(situation: str, cues: dict | None = None) -> bool:
    r, cues = spec()["safety_rules"], cues or {}
    return situation in r["always_yes_situations"] or any(cues.get(c) for c in r["yes_if_cue"])


_WORD = re.compile(r"[a-zA-Z]+")


def hindi_or_hinglish(romanised: str, english_words: set[str]) -> str:
    """Helper for labelling romanised Hindi-grammar text: counts English words outside the loanword whitelist.
    `english_words` is the set of tokens the author marks as English (the LLM drafting step supplies it)."""
    rule = spec()["language_rule"]
    loan = set(rule["loanwords_counted_as_hindi"])
    other = [w for w in _WORD.findall(romanised.lower()) if w in english_words and w not in loan]
    return "hinglish" if len(other) >= rule["hinglish_min_other_english_words"] else "hindi"


def label_row(row: dict) -> dict:
    """Fill gold labels for one script row (see data/scripts/README.md for the schema)."""
    s, cues = row["situation"], row.get("cues") or {}
    if row.get("language") not in LANGUAGES:
        raise ValueError(f"{row.get('id')}: unknown language {row.get('language')!r}")
    return {**row, "action": derive_action(s, cues), "safety": derive_safety(s, cues)}


# --------------------------------------------------------------------------- self-test
def _selftest() -> None:
    sp = spec()
    assert len(SITUATIONS) == 12 and len(ACTIONS) == 6 and len(LANGUAGES) == 3, (SITUATIONS, ACTIONS, LANGUAGES)
    assert set(sp["action_rules"]) == set(SITUATIONS), "every situation needs an action rule"
    reachable = set()
    for s, r in sp["action_rules"].items():
        assert r["default"] in ACTIONS, (s, r["default"])
        reachable.add(r["default"])
        for o in r["overrides"]:
            assert o["then"] in ACTIONS, (s, o)
            reachable.add(o["then"])
    assert reachable == set(ACTIONS), f"unreachable actions: {set(ACTIONS) - reachable}"

    cases = [  # (situation, cues, action, safety) — the seed utterances and the edge cases from the spec
        ("customer_unreachable", {}, "call_customer", False),                        # "gate band hai, phone nahi utha raha"
        ("customer_unreachable", {"attempts": 3, "waited_min": 12}, "contact_support", False),
        ("customer_unreachable", {"rider_waiting": True, "waited_min": 2}, "wait", False),
        ("restaurant_delay", {"waited_min": 20}, "request_reassign", False),          # "20 minute aur lagega"
        ("restaurant_delay", {"waited_min": 5}, "wait", False),
        ("food_spilled_or_damaged", {"near_restaurant": True}, "return_to_restaurant", False),
        ("order_issue_at_pickup", {"left_restaurant": True, "near_restaurant": True}, "return_to_restaurant", False),
        ("order_issue_at_pickup", {"still_at_restaurant": True, "fixable": True}, "wait", False),
        ("vehicle_breakdown", {}, "request_reassign", False),                        # puncture, no shop nearby
        ("vehicle_breakdown", {"fix_minutes": 5}, "contact_support", False),
        ("harassment_or_threat", {}, "sos", True),                                   # "dhamki de rahe hain"
        ("harassment_or_threat", {"rider_left_safe": True}, "contact_support", True),  # safety stays yes
        ("accident_or_injury", {"minor": True, "rider_ok": True}, "contact_support", True),
        ("road_blocked_or_weather", {}, "wait", False),                              # "aage accident hua hai" (not the rider)
        ("road_blocked_or_weather", {"dangerous_place": True, "cannot_proceed": True}, "contact_support", True),
        ("other_or_unclear", {"medical_emergency": True}, "sos", True),             # spec 0.4: medical emergency -> sos
        ("payment_issue", {"customer_fetching_money": True}, "wait", False),
    ]
    for s, cues, a, safe in cases:
        assert derive_action(s, cues) == a, (s, cues, derive_action(s, cues), a)
        assert derive_safety(s, cues) == safe, (s, cues)

    assert hindi_or_hinglish("customer phone nahi utha raha", {"customer", "phone"}) == "hindi"
    assert hindi_or_hinglish("customer is not responding, kya karun", {"customer", "is", "not", "responding"}) == "hinglish"
    print(f"labels self-test OK: {len(SITUATIONS)} situations, {len(ACTIONS)} actions (all reachable), "
          f"{len(LANGUAGES)} languages, {len(cases)} rule cases")


if __name__ == "__main__":
    _selftest()


In [ ]:
%%writefile /kaggle/working/rider-voice-space/src/product.py
"""The product layer: model answers -> what the rider sees. The model suggests; the rider confirms with one tap.

Rules (fixed, not learned):
  1. SOS - the emergency card + human escalation - when the SOS level fires (src/decide.alert_level).
  2. CHECK-IN - "Are you safe?" with two buttons, [I'm safe] and [Get help] - when the check-in level fires. If the
     rider does not answer within `checkin_timeout_s` (default 30 s) it escalates to SOS by itself, so a check-in
     can never silently swallow a real emergency. The normal action suggestion is shown under it.
  3. Otherwise one big button for the top action, plus the runner-up and "talk to support".
  4. If the top action's probability < low_conf, or the situation is other_or_unclear, the top two actions are shown
     as equal choices plus support instead of a single suggestion.
Nothing is ever executed without a tap (except the check-in's timeout escalation, which errs towards help).

Runs calibrated before round 3 have no sos_tau/checkin_tau: they keep the single-level behaviour (SOS whenever
p(safety) >= safety_tau, a safety situation is top-1, or the top action is sos).

    python src/product.py     # self-test
"""

from __future__ import annotations

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from src.decide import alert_level, safety_scores  # noqa: E402

DEFAULTS = {"safety_tau": 0.2, "low_conf": 0.6, "checkin_timeout_s": 30}


def _suggestion(answers: dict, th: dict) -> dict:
    act_p = answers["action"]["probabilities"]
    ranked = sorted((a for a in act_p if act_p[a] > 0), key=act_p.get, reverse=True) or sorted(act_p, key=act_p.get, reverse=True)
    sit = answers["situation"]["choice"]
    support = [] if "contact_support" in ranked[:2] else ["contact_support"]
    low = act_p[ranked[0]] < th["low_conf"]
    if low or sit == "other_or_unclear":
        return {"kind": "choose", "primary": None, "alternatives": ranked[:2] + support,
                "reasons": ["low confidence" if low else "unclear situation"]}
    return {"kind": "suggest", "primary": answers["action"]["choice"], "alternatives": [a for a in ranked if a != answers["action"]["choice"]][:1] + support,
            "reasons": []}


def card(answers: dict, thresholds: dict | None = None) -> dict:
    th = {**DEFAULTS, **(thresholds or {})}
    level = alert_level(answers, th)
    sc = {k: round(v, 4) for k, v in safety_scores(answers).items()}
    if level == "sos":
        return {"kind": "sos", "primary": "sos", "alternatives": ["contact_support"], "escalate_to_human": True,
                "reasons": [f"p(sos)={sc['p_sos']:.2f}" if "sos_tau" in th else "safety flag"], "scores": sc,
                "requires_tap": True}
    sug = _suggestion(answers, th)
    if level == "checkin":
        return {"kind": "checkin", "question": "Are you safe?", "buttons": ["i_am_safe", "get_help"],
                "auto_escalate_after_s": th["checkin_timeout_s"], "escalate_to_human": False,
                "then": sug, "primary": sug["primary"], "alternatives": sug["alternatives"],
                "reasons": [f"safety score {sc['checkin_score']:.2f} >= {th['checkin_tau']:.2f}"], "scores": sc,
                "requires_tap": True}
    return {**sug, "escalate_to_human": False, "scores": sc, "requires_tap": True}


def _ans(sit, act, p_act, p_safe, p_sit_safe=0.0):
    acts = ["wait", "call_customer", "contact_support", "request_reassign", "return_to_restaurant", "sos"]
    rest = (1 - p_act) / (len(acts) - 1)
    sits = {"customer_unreachable": 0.0, "accident_or_injury": p_sit_safe, "harassment_or_threat": 0.0, sit: 0.0}
    sits[sit] = sits.get(sit, 0.0) + max(0.0, 1 - p_sit_safe) if sit not in ("accident_or_injury", "harassment_or_threat") else 1.0
    return {"situation": {"choice": sit, "probabilities": sits},
            "action": {"choice": act, "probabilities": {a: (p_act if a == act else rest) for a in acts}},
            "safety": {"noul": p_safe}}


def _selftest():
    # legacy single level
    c = card(_ans("customer_unreachable", "call_customer", 0.8, 0.02))
    assert c["kind"] == "suggest" and c["primary"] == "call_customer" and "contact_support" in c["alternatives"]
    assert card(_ans("customer_unreachable", "call_customer", 0.8, 0.5))["kind"] == "sos"           # safety prob
    assert card(_ans("harassment_or_threat", "contact_support", 0.9, 0.01))["kind"] == "sos"        # situation
    assert card(_ans("vehicle_breakdown", "sos", 0.7, 0.01))["kind"] == "sos"                       # action
    c = card(_ans("vehicle_breakdown", "request_reassign", 0.4, 0.01))
    assert c["kind"] == "choose" and c["primary"] is None and c["alternatives"][0] == "request_reassign"
    assert card(_ans("other_or_unclear", "contact_support", 0.9, 0.01))["kind"] == "choose"
    # two-level
    th = {"sos_tau": 0.4, "checkin_tau": 0.2}
    assert card(_ans("harassment_or_threat", "sos", 0.7, 0.9), th)["kind"] == "sos"
    c = card(_ans("accident_or_injury", "contact_support", 0.8, 0.6), th)         # minor fall, rider fine
    assert c["kind"] == "checkin" and c["auto_escalate_after_s"] == 30 and c["then"]["primary"] == "contact_support"
    c = card(_ans("payment_issue", "wait", 0.8, 0.05), th)
    assert c["kind"] == "suggest" and c["primary"] == "wait"
    assert card(_ans("payment_issue", "wait", 0.8, 0.25), th)["kind"] == "checkin"  # borderline -> ask, don't alarm
    assert all(card(_ans(s, "wait", 0.9, p), th)["requires_tap"] for s in ("payment_issue", "accident_or_injury") for p in (0.0, 0.9))
    print("product self-test OK")


if __name__ == "__main__":
    _selftest()


In [ ]:
%%writefile /kaggle/working/rider-voice-space/src/rider_copy.py
"""What the rider sees and hears back for each product card (illustrative UI copy only).

src/product.py decides the card kind/primary action; it has no rider-facing strings, so these are the demo
defaults. Nothing here is executed: no call, SMS or ticket is actually placed.
"""

from __future__ import annotations

# key -> {lang: (instruction, spoken)}, gloss, buttons, status, ops
COPY = {
    "wait": {
        "text": {
            "hinglish": ("Ruko, customer ko bata diya hai", "Ruko, customer ko bata diya hai. Paanch minute ka timer chalu hai."),
            "hindi": ("रुको, कस्टमर को बता दिया है", "रुको, कस्टमर को बता दिया है। पाँच मिनट का टाइमर चालू है।"),
            "english": ("Hold on, customer has been notified", "Hold on, the customer has been notified. A five minute timer is running."),
        },
        "gloss": "Hold on, customer has been notified",
        "buttons": ["Waiting ✓", "Problem"],
        "status": "Customer notified · 5 min timer",
        "ops": "CUSTOMER SMS “Your rider is waiting at the drop point” · OPS: 5-min wait timer on {sit}",
    },
    "call_customer": {
        "text": {
            "hinglish": ("Customer ko call kar rahe hain", "Customer ko call kar rahe hain, line pe rahiye."),
            "hindi": ("कस्टमर को कॉल कर रहे हैं", "कस्टमर को कॉल कर रहे हैं, लाइन पर रहिए।"),
            "english": ("Calling the customer for you", "Calling the customer for you, please stay on the line."),
        },
        "gloss": "Calling the customer for you",
        "buttons": ["Connecting…", "Cancel"],
        "status": "Masked call placed",
        "ops": "OPS: masked call rider ↔ customer · log “{sit} → call customer”",
    },
    "contact_support": {
        "text": {
            "hinglish": ("Support se jod rahe hain", "Support se jod rahe hain. Aapka voice note bhej diya hai."),
            "hindi": ("सपोर्ट से जोड़ रहे हैं", "सपोर्ट से जोड़ रहे हैं। आपका वॉइस नोट भेज दिया है।"),
            "english": ("Connecting you to support", "Connecting you to support. Your voice note has been sent."),
        },
        "gloss": "Connecting you to support",
        "buttons": ["Talk now", "Later"],
        "status": "Ticket raised with your voice note",
        "ops": "OPS TICKET: {sit} → talk to support · voice note attached",
    },
    "request_reassign": {
        "text": {
            "hinglish": ("Order dusre rider ko de rahe hain", "Order dusre rider ko de rahe hain. Confirm karne ke liye tap karein."),
            "hindi": ("ऑर्डर दूसरे राइडर को दे रहे हैं", "ऑर्डर दूसरे राइडर को दे रहे हैं। पक्का करने के लिए टैप करें।"),
            "english": ("Reassigning the order to another rider", "Reassigning the order to another rider. Tap to confirm."),
        },
        "gloss": "Reassigning the order to another rider",
        "buttons": ["Confirm", "Keep order"],
        "status": "Reassign request pending your tap",
        "ops": "OPS: reassign request · {sit}",
    },
    "return_to_restaurant": {
        "text": {
            "hinglish": ("Restaurant wapas jaiye", "Restaurant wapas jaiye, unko bata diya hai."),
            "hindi": ("रेस्टोरेंट वापस जाइए", "रेस्टोरेंट वापस जाइए, उनको बता दिया है।"),
            "english": ("Head back to the restaurant", "Head back to the restaurant, they have been told."),
        },
        "gloss": "Head back to the restaurant",
        "buttons": ["Navigate", "Problem"],
        "status": "Restaurant informed",
        "ops": "RESTAURANT + OPS: rider returning · {sit}",
    },
    "choose": {
        "text": {
            "hinglish": ("Kya karna hai? Ek chuniye", "Mujhe pakka nahi pata. Kya karna hai, ek chuniye."),
            "hindi": ("क्या करना है? एक चुनिए", "मुझे पक्का नहीं पता। क्या करना है, एक चुनिए।"),
            "english": ("Not sure — pick what to do", "I'm not sure. Please pick what to do."),
        },
        "gloss": "Not sure — pick what to do",
        "buttons": [],  # filled from the card's alternatives
        "status": "Low confidence · rider decides",
        "ops": "OPS: nothing sent until the rider taps · {sit}",
    },
    "checkin": {
        "text": {
            "hinglish": ("Aap theek ho?", "Aap theek ho? Tees second mein jawab nahi aaya to support ko bhejenge."),
            "hindi": ("आप ठीक हो?", "आप ठीक हो? तीस सेकंड में जवाब नहीं आया तो सपोर्ट को भेजेंगे।"),
            "english": ("Are you safe?", "Are you safe? If you don't reply in thirty seconds, we'll alert support."),
        },
        "gloss": "Are you safe?",
        "buttons": ["I'm safe", "Need help"],
        "status": "No reply in 30 s → escalate to support",
        "ops": "OPS: safety watch flag · auto-escalates to support after 30 s",
    },
    "sos": {
        "text": {
            "hinglish": ("Madad aa rahi hai", "Madad aa rahi hai. Aapki location support aur emergency contact ko bhej di hai."),
            "hindi": ("मदद आ रही है", "मदद आ रही है। आपकी लोकेशन सपोर्ट और इमरजेंसी कॉन्टैक्ट को भेज दी है।"),
            "english": ("Help is on the way", "Help is on the way. Your location has been shared with support and your emergency contact."),
        },
        "gloss": "Help is on the way",
        "buttons": ["Cancel (10 s)"],
        "status": "Live location shared with support + emergency contact",
        "ops": "SUPPORT + EMERGENCY CONTACT: live location, voice note, {sit}",
    },
}

ACTION_BTN = {"wait": "Wait", "call_customer": "Call customer", "contact_support": "Support",
              "request_reassign": "Reassign", "return_to_restaurant": "Go back", "sos": "SOS"}


def rider_view(card: dict, answers: dict, sit_label: str) -> dict:
    kind = card.get("kind")
    key = kind if kind in ("sos", "checkin", "choose") else (card.get("primary") or answers["action"]["choice"])
    c = COPY.get(key, COPY["contact_support"])
    lang = answers.get("language", {}).get("choice", "english")
    lang = lang if lang in c["text"] else "english"
    line, spoken = c["text"][lang]
    buttons = c["buttons"] or [ACTION_BTN.get(a, a) for a in card.get("alternatives", [])][:2]
    then = None
    if kind == "checkin" and card.get("primary"):
        then = COPY.get(card["primary"], {}).get("gloss")
    return {"key": key, "kind": kind, "lang": lang, "line": line, "gloss": c["gloss"], "spoken": spoken,
            "buttons": buttons, "status": c["status"], "ops": c["ops"].format(sit=sit_label), "then": then}


In [ ]:
%%writefile /kaggle/working/rider-voice-space/src/request.py
"""Build the one request every clip is scored with: a state plus four typed questions, in the
Decision Index format (`choice` with `criteria`, `noul` for yes/no).

The prompt text the backbone reads is rendered by `mmclm.joint.prompt_text` from nirnaya-inference when it is
importable. `render_prompt_text` below reproduces the same layout so the repo can be tested without it.

    python src/request.py         # prints the request and the rendered prompt text
"""

from __future__ import annotations

import json

try:
    from src.labels import spec
except ImportError:  # run as a script from the repo root
    from labels import spec

INSTRUCTION = "Answer each question with the number of the correct option only."


def build_request() -> tuple[str, dict]:
    sp = spec()
    return sp["state"], sp["questions"]


def _shown(questions: dict) -> list[tuple[dict, list[int]]]:
    """Questions in the joint-head shape: options in spec order (no shuffling — features are cached)."""
    shown = []
    for q in questions.values():
        opts = ["yes", "no"] if q["type"] == "noul" else list(q["criteria"].values())
        shown.append(({"instruction": q["instructions"], "type": q["type"], "options": opts}, list(range(len(opts)))))
    return shown


def render_prompt_text(state: str, questions: dict) -> str:
    try:
        from mmclm import joint

        return joint.prompt_text(state, _shown(questions))
    except ImportError:
        blocks = []
        for i, (q, perm) in enumerate(_shown(questions)):
            lines = "\n".join(f"{j + 1}. {q['options'][p]}" for j, p in enumerate(perm))
            blocks.append(f"[Q{i + 1}] {q['instruction']}".rstrip() + "\n" + lines)
        return f"{state.strip()}\n\nQuestions:\n" + "\n".join(blocks) + f"\n\n{INSTRUCTION}"


if __name__ == "__main__":
    state, questions = build_request()
    print(json.dumps({"state": state, "questions": questions}, indent=1, ensure_ascii=False)[:600], "...\n")
    print(render_prompt_text(state, questions))


In [ ]:
import base64
import io
import zipfile
from pathlib import Path

# Held-out synthetic test clips (not real rider recordings), bundled so the
# notebook does not need the private GitHub repo or a Space snapshot.
EXAMPLES_B64 = """
...
"""

dest = Path("/kaggle/working/rider-voice-space/examples")
dest.mkdir(parents=True, exist_ok=True)
raw = base64.b64decode("".join(EXAMPLES_B64.split()))
with zipfile.ZipFile(io.BytesIO(raw)) as zf:
    zf.extractall(dest)
wavs = sorted(p.name for p in dest.glob("*.wav"))
print(f"examples: {len(wavs)} clips, zip was 1028392 bytes", flush=True)
for name in wavs:
    print(" ", name, flush=True)
assert len(wavs) == 9, wavs


In [ ]:
import os
import sys
import time
from pathlib import Path

from huggingface_hub import snapshot_download
from IPython.display import HTML, Markdown, display

APP = Path("/kaggle/working/rider-voice-space")
TMP = Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else Path("/tmp")
RUN_DIR = TMP / "m1v3"
MODEL_DIR = TMP / "nirnaya-flash-1.0"
NIRNAYA_DIR = TMP / "nirnaya-inference"

os.chdir(APP)
os.environ["MOCK"] = "0"
os.environ["RUN_DIR"] = str(RUN_DIR)
os.environ["HF_MODEL_REPO"] = "sahildl/rider-voice-decisions-m1v3"
os.environ["MODEL_DIR"] = str(MODEL_DIR)
os.environ["MODEL_ID"] = "Darkrider/nirnaya-flash-1.0"
os.environ["PRECISION"] = "fp16"
os.environ["NIRNAYA_DIR"] = str(NIRNAYA_DIR)
os.environ["THRESHOLDS"] = str(APP / "thresholds.json")

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not token:
    raise SystemExit("HF_TOKEN is missing. Re-run the secrets cell.")

print("Downloading private m1v3 head ->", RUN_DIR, flush=True)
snapshot_download(
    repo_id="sahildl/rider-voice-decisions-m1v3",
    repo_type="model",
    local_dir=str(RUN_DIR),
    token=token,
    allow_patterns=[
        "head.pt",
        "config.json",
        "t_entropy.json",
        "thresholds.json",
        "calibration.json",
    ],
)
print("head files:", sorted(p.name for p in RUN_DIR.iterdir() if p.is_file()), flush=True)
if not (RUN_DIR / "head.pt").exists():
    raise SystemExit("head.pt missing after download. Check that HF_TOKEN can read sahildl/rider-voice-decisions-m1v3.")

print("Downloading public backbone Darkrider/nirnaya-flash-1.0 (~16 GB) ->", MODEL_DIR, flush=True)
snapshot_download(
    repo_id="Darkrider/nirnaya-flash-1.0",
    local_dir=str(MODEL_DIR),
    token=token,
)
print("backbone ready", flush=True)

# Import after env vars are set. app._boot() loads the engine and, on CUDA, warms it in fp16.
if str(APP) not in sys.path:
    sys.path[:0] = [str(APP)]
import app as rider_app  # noqa: E402

print(
    "ENGINE loaded:", rider_app.ENGINE is not None,
    "| MOCK:", rider_app.MOCK,
    "| cuda warmup done if ENGINE is set",
    flush=True,
)
if rider_app.ENGINE is None:
    raise SystemExit(f"Real engine failed to load: {rider_app.ENGINE_ERROR}")

# Public Gradio link. prevent_thread_lock=True so we can print the URL, then we
# block this cell so the server stays up.
_app, local_url, share_url = rider_app.demo.launch(
    theme=rider_app.theme,
    css=rider_app.CSS,
    js=rider_app.DARK_JS,
    head=rider_app.HEAD,
    footer_links=[],
    server_name="0.0.0.0",
    server_port=int(os.environ.get("PORT", "7860")),
    allowed_paths=[str(APP / "examples"), str(APP / "assets"), str(APP)],
    ssr_mode=False,
    show_error=True,
    share=True,
    inline=False,
    prevent_thread_lock=True,
)

url = share_url or getattr(rider_app.demo, "share_url", None) or ""
display(Markdown(f"# Public live demo\n\n**[{url}]({url})**\n\nOpen this while the cell is still running."))
display(HTML(
    f'<p style="font-size:28px;font-family:Space Grotesk,sans-serif;margin:12px 0">'
    f'<a href="{url}">{url}</a></p>'
))
print("=" * 72, flush=True)
print("PUBLIC LIVE DEMO LINK", flush=True)
print(url, flush=True)
print("local:", local_url, flush=True)
print("Keep this cell running. Stop the Kaggle session when you are done.", flush=True)
print("=" * 72, flush=True)

while True:
    time.sleep(60)


## When you are done

The public link only works **while this Kaggle session is running**. Gradio share links also expire after about **72 hours**, even if the session is still up.

**Stop the session** when you finish (Kaggle notebook UI → Stop session). Closing the browser tab is not enough, and leaving a T4 running will burn your free weekly GPU quota.